# Recycling Points in Berlin – Data Extraction & Normalization

This notebook:
- Extracts recycling locations from OpenStreetMap using OSMnx
- Interprets granular `recycling:*` tags into human-readable categories
- Cleans and consolidates noisy OSM metadata
- Produces a user-facing, analysis-ready GeoDataFrame

Design principles:
- Do not invent information
- Prefer structured tags over free text
- Preserve provenance where ambiguity exists


In [ ]:
# install libraries
#%pip install osmnx geopandas pandas
#%pip install geopy
#%pip install --upgrade certifi


Note: you may need to restart the kernel to use updated packages.


In [263]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np

from geopy.geocoders import Nominatim
from shapely.geometry import Point, Polygon
from time import sleep

import re

In [264]:
# Give me all places tagged as OpenStreetMap – Points of interest tagged as amenity=recycling.
# tags filter for only features with 

tags = {"amenity": "recycling"}

In [265]:
# ✅ Enables caching: Speeds up repeated queries
# 🖥 Logs details to the console (helpful for debugging)
"""ox.settings.use_cache = True
ox.settings.log_console = True"""

'ox.settings.use_cache = True\nox.settings.log_console = True'

## Data Source Strategy

Initial extraction from OpenStreetMap was performed once and saved to disk.
To ensure reproducibility and avoid repeated API calls, the notebook
continues from the persisted GeoJSON below.

In [266]:
# Fetch Supermarkets from Berlin from OSM using the tag "shop=supermarket"
"""df = ox.features.features_from_place("Berlin, Germany", tags=tags)"""

'df = ox.features.features_from_place("Berlin, Germany", tags=tags)'

In [267]:
"""df = df.reset_index()
df.head()"""

'df = df.reset_index()\ndf.head()'

In [268]:
"""df.to_file("../sources/raw_recycling_points.geojson", driver="GeoJSON")"""

'df.to_file("../sources/raw_recycling_points.geojson", driver="GeoJSON")'

In [269]:
"""df.to_csv("../sources/raw_recycling_points.csv", index=False)"""

'df.to_csv("../sources/raw_recycling_points.csv", index=False)'

## Load Persisted OSM Extract

To ensure reproducibility and avoid repeated OpenStreetMap queries,
the analysis continues from a previously saved GeoJSON file.


In [270]:
# Load persisted OSM extract to avoid repeated network queries

df = gpd.read_file("../sources/raw_recycling_points.geojson")
df.head()


,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,industrial,building:colour,building:material,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry
0,node,26867409,recycling,2024-07-05,Berlin Recycling,yes,container,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.29683 52.50133)
1,node,254985049,recycling,2025-03-11,None,None,container,Deutsches Rotes Kreuz,Q694104,de:Deutsches Rotes Kreuz,...,None,None,None,None,None,None,None,None,None,POINT (13.32054 52.484)
2,node,262212828,recycling,2025-01-20,None,yes,container,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.48568 52.51902)
3,node,267093410,recycling,2021-09-28,None,yes,container,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.59289 52.50846)
4,node,272619379,recycling,2025-05-07,None,yes,container,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.45563 52.52183)


In [271]:
gdf = df.copy()

In [272]:
# Display basic info

print(f"Number of recycling points entries fetched: {len(gdf)}")
gdf.head(3)

Number of recycling points entries fetched: 2842


,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,industrial,building:colour,building:material,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry
0,node,26867409,recycling,2024-07-05,Berlin Recycling,yes,container,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.29683 52.50133)
1,node,254985049,recycling,2025-03-11,None,None,container,Deutsches Rotes Kreuz,Q694104,de:Deutsches Rotes Kreuz,...,None,None,None,None,None,None,None,None,None,POINT (13.32054 52.484)
2,node,262212828,recycling,2025-01-20,None,yes,container,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.48568 52.51902)


In [273]:
for col in gdf.columns:
    print(col)

element
id
amenity
check_date:recycling
operator
recycling:glass_bottles
recycling_type
brand
brand:wikidata
brand:wikipedia
check_date
description
note
recycling:clothes
location
mapillary
survey:date
source
access
recycling:glass
ref
recycling:shoes
wheelchair
created_by
recycling:glass_bottles:colour
opening_hours
operator:wikidata
name
addr:city
addr:country
addr:street
addr:housenumber
addr:postcode
addr:suburb
recycling:batteries
recycling:cans
recycling:paper
recycling:scrap_metal
website
check_date:opening_hours
addr:floor
covered
indoor
level
material
recycling:PET
recycling:cardboard
recycling:plastic
recycling:plastic_bottles
recycling:plastic_packaging
count
recycling:christmas_trees
operator:phone
contact:email
contact:phone
recycling:green_waste
colour
operator:short
contact:website
obstacle:parking
source:amenity
recycling:beverage_cartons
recycling:books
recycling:cartons
recycling:chipboard
recycling:cork
recycling:garden_waste
recycling:newspaper
recycling:paper_packa

In [274]:
empty_cols = []      # columns with no data
non_empty_cols = []  # columns with data

for col in gdf.columns:
    uniques = gdf[col].unique()
    
    if len(uniques) > 0:
        non_empty_cols.append(col)
        print(f"\n==== {col} ====")
        print(uniques)
    else:
        empty_cols.append(col)

print("\n\nColumns with NO data:")
print(empty_cols)



==== element ====
['node' 'way']

==== id ====
[  26867409  254985049  262212828 ... 1451298080 1451319774 1452164030]

==== amenity ====
['recycling']

==== check_date:recycling ====
<DatetimeArray>
['2024-07-05 00:00:00', '2025-03-11 00:00:00', '2025-01-20 00:00:00',
 '2021-09-28 00:00:00', '2025-05-07 00:00:00', '2023-08-01 00:00:00',
 '2024-11-23 00:00:00',                 'NaT', '2023-04-12 00:00:00',
 '2024-08-24 00:00:00',
 ...
 '2025-06-14 00:00:00', '2025-06-12 00:00:00', '2025-07-02 00:00:00',
 '2025-10-09 00:00:00', '2025-03-24 00:00:00', '2025-04-07 00:00:00',
 '2025-07-11 00:00:00', '2025-10-27 00:00:00', '2025-11-28 00:00:00',
 '2025-09-22 00:00:00']
Length: 611, dtype: datetime64[ms]

==== operator ====
['Berlin Recycling' None 'Rhenus' 'Karl Meyer Rohstoffverwertung' 'Humana'
 'Karl Meyer AG' 'Deutsches Rotes Kreuz' 'BSR' 'Bera Textilrecycling'
 'Loop Textil Recycling Nord' 'Deutsches Rotes Kreuz e.V.'
 'Fa. Cavtex (www.cavtex.de)' 'TEXAID Deutschland' 'berlin recyclin

## Understanding OSM Recycling Tags

OpenStreetMap models recycling using a hierarchical tag structure:

- `recycling:<item>` → whether an item is accepted (yes / no / customers)
- `recycling:<item>:<subtype>` → item-specific details (e.g. glass color)

Examples:
- `recycling:glass_bottles = yes`
- `recycling:glass_bottles:colour = white;green`

The logic below separates:
- base materials
- material subtypes
so they can be interpreted consistently.


In [275]:
recycling_cols = [
    c for c in gdf.columns 
    if c.startswith("recycling:") and c.count(":") == 1
]
recycling_cols_sub = [
    c for c in gdf.columns 
    if c.startswith("recycling:") and c.count(":") == 2
]



In [276]:
def dedup_preserve_order(lst):
    seen = set()
    ordered = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            ordered.append(item)
    return ordered

In [277]:
def extract_recycling(row):
    """
    Parse OSM recycling tags into three explicit categories:

    - accepted: materials explicitly accepted at this location
    - not_accepted: materials explicitly rejected
    - other: ambiguous or unclear acceptance

    Design choices:
    - Preserve order of appearance (OSM tagging is human-entered)
    - Do not collapse subtype information (e.g. glass colors)
    - Avoid inferring acceptance where tags are ambiguous
    """
    accepted = []
    not_accepted = []
    other = []

    # 1. Base items (recycling:glass_bottles)
    for col in recycling_cols:
        base = col.split(":")[1]
        val = row[col]

        if isinstance(val, str):
            if val == "yes" or "yes" in val.split(";"):
                accepted.append(base)
            elif val == "no" or "no" in val.split(";"):
                not_accepted.append(base)
            elif val == "customers":
                accepted.append(val + '_' + base)
            else:
                other.append(val + '_' + base)

    # 2. Sub-items (recycling:glass_bottles:colour)
    for col in recycling_cols_sub:
        _, base, subtype = col.split(":")

        val = row[col]

        if not isinstance(val, str):
            continue

        # Case: color list: "white;green;brown"
        colors = [x.strip() for x in val.split(";")]

        # Case: value is "yes" meaning "all colors allowed" (rare)
        if val == "yes":
            accepted.append(f"{base}_unknown")
            continue
        elif val == "no":
            not_accepted.append(f"{base}_unknown")
            continue

        # Normal case: add each color explicitly
        for color in colors:
            if color:  # avoid empty strings
                accepted.append(f"{base}_{color}")

    # Remove duplicates but preserve order
    accepted = dedup_preserve_order(accepted)
    not_accepted = dedup_preserve_order(not_accepted)
    other = dedup_preserve_order(other)

    return accepted, not_accepted, other

"""    # Remove duplicates but preserve order
    seen = set()
    ordered = []

    for item in accepted:
        if item not in seen:
            seen.add(item)
            ordered.append(item)

    return ordered"""


#gdf["recycling_items_flat"] = gdf.apply(extract_recycling_flat, axis=1)


'    # Remove duplicates but preserve order\n    seen = set()\n    ordered = []\n\n    for item in accepted:\n        if item not in seen:\n            seen.add(item)\n            ordered.append(item)\n\n    return ordered'

In [278]:
gdf["accepted_recycling_items"], gdf["not_accepted_recycling_items"], gdf["other_recycling_items"] = zip(*gdf.apply(extract_recycling, axis=1))


In [279]:
gdf.head()

,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
0,node,26867409,recycling,2024-07-05,Berlin Recycling,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],[],[]
1,node,254985049,recycling,2025-03-11,None,None,container,Deutsches Rotes Kreuz,Q694104,de:Deutsches Rotes Kreuz,...,None,None,None,None,None,None,POINT (13.32054 52.484),[clothes],[],[]
2,node,262212828,recycling,2025-01-20,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.48568 52.51902),[glass_bottles],[],[]
3,node,267093410,recycling,2021-09-28,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.59289 52.50846),[glass_bottles],[],[]
4,node,272619379,recycling,2025-05-07,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.45563 52.52183),[glass_bottles],[],[]


In [280]:
# Replace empty lists with NA
new_cols = ["accepted_recycling_items", "not_accepted_recycling_items", "other_recycling_items"]

for col in new_cols:
    gdf[col] = gdf[col].apply(lambda v: pd.NA if isinstance(v, list) and len(v) == 0 else v)


In [281]:
gdf.head()

,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
0,node,26867409,recycling,2024-07-05,Berlin Recycling,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,<NA>
1,node,254985049,recycling,2025-03-11,None,None,container,Deutsches Rotes Kreuz,Q694104,de:Deutsches Rotes Kreuz,...,None,None,None,None,None,None,POINT (13.32054 52.484),[clothes],<NA>,<NA>
2,node,262212828,recycling,2025-01-20,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,<NA>
3,node,267093410,recycling,2021-09-28,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,<NA>
4,node,272619379,recycling,2025-05-07,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,<NA>


In [282]:
gdf.loc[
    (gdf["recycling:glass_bottles"] == "yes") &
    (gdf["recycling:glass_bottles:colour"].notna()),
    ["name", "accepted_recycling_items", "not_accepted_recycling_items", "other_recycling_items", "recycling:glass_bottles", "recycling:glass_bottles:colour"]]

,name,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items,recycling:glass_bottles,recycling:glass_bottles:colour
31,None,"[glass_bottles, glass_bottles_white, glass_bot...",<NA>,<NA>,yes,white;green;brown
44,None,"[glass_bottles, glass_bottles_white, glass_bot...",<NA>,<NA>,yes,white;green;brown
46,None,"[glass_bottles, glass_bottles_green, glass_bot...",<NA>,<NA>,yes,green;white
47,None,"[glass_bottles, glass_bottles_white, glass_bot...",<NA>,<NA>,yes,white;green;brown
48,None,"[glass_bottles, clothes, glass_bottles_green, ...",<NA>,<NA>,yes,green;white
...,...,...,...,...,...,...
2219,None,"[glass_bottles, glass_bottles_brown, glass_bot...",<NA>,<NA>,yes,brown;green;white
2420,None,"[glass_bottles, glass_bottles_white]",<NA>,<NA>,yes,white
2422,None,"[glass_bottles, glass_bottles_brown]",<NA>,<NA>,yes,brown
2743,None,"[glass_bottles, glass_bottles_clear]",<NA>,<NA>,yes,clear


In [283]:
gdf_clean = gdf.drop(columns=recycling_cols + recycling_cols_sub)
gdf_clean.head()

,element,id,amenity,check_date:recycling,operator,recycling_type,brand,brand:wikidata,brand:wikipedia,check_date,...,building:material,roof:colour,roof:material,roof:shape,disused:parking,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
0,node,26867409,recycling,2024-07-05,Berlin Recycling,container,None,None,None,None,...,None,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,<NA>
1,node,254985049,recycling,2025-03-11,None,container,Deutsches Rotes Kreuz,Q694104,de:Deutsches Rotes Kreuz,2024-03-10,...,None,None,None,None,None,None,POINT (13.32054 52.484),[clothes],<NA>,<NA>
2,node,262212828,recycling,2025-01-20,None,container,None,None,None,None,...,None,None,None,None,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,<NA>
3,node,267093410,recycling,2021-09-28,None,container,None,None,None,None,...,None,None,None,None,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,<NA>
4,node,272619379,recycling,2025-05-07,None,container,None,None,None,None,...,None,None,None,None,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,<NA>


In [284]:
gdf_clean.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2842 entries, 0 to 2841
Data columns (total 89 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   element                       2842 non-null   object        
 1   id                            2842 non-null   int64         
 2   amenity                       2842 non-null   object        
 3   check_date:recycling          992 non-null    datetime64[ms]
 4   operator                      840 non-null    object        
 5   recycling_type                2803 non-null   object        
 6   brand                         260 non-null    object        
 7   brand:wikidata                241 non-null    object        
 8   brand:wikipedia               95 non-null     object        
 9   check_date                    152 non-null    object        
 10  description                   21 non-null     object        
 11  note                  

## Understanding the data better before grouping

In [285]:
gdf[gdf["recycling:paper"] == "customers"]

,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
2700,way,543319990,recycling,NaT,None,None,centre,None,None,None,...,None,None,None,None,None,None,"POLYGON ((13.55084 52.50918, 13.55038 52.50921...","[customers_paper, customers_pallets]",<NA>,<NA>


In [286]:
gdf[gdf["recycling:clothes"] == "no"]

,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
302,node,737216262,recycling,2023-04-16,Karl Meyer Rohstoffverwertung GmbH,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.42943 52.54495),[glass_bottles],"[clothes, cans, paper, plastic]",<NA>
905,node,4043265018,recycling,2023-03-11,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.59996 52.43839),[glass_bottles],"[clothes, glass, cans, paper, plastic]",<NA>
906,node,4043268713,recycling,2023-03-11,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.59059 52.44247),[glass_bottles],"[clothes, glass, cans, paper, plastic]",<NA>
908,node,4058941748,recycling,2024-12-12,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.59363 52.44705),[glass_bottles],"[clothes, glass, cans, paper, plastic]",<NA>
944,node,4448477388,recycling,2024-03-04,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.34946 52.54836),[glass_bottles],"[clothes, glass, cans, paper]",<NA>
985,node,5070021874,recycling,2022-09-12,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.60135 52.44183),[glass_bottles],"[clothes, glass, cans, paper, plastic]",<NA>
1019,node,5792096602,recycling,2024-05-30,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.37481 52.46859),[glass_bottles],"[clothes, batteries, cans, paper, scrap_metal,...",<NA>
1635,node,9741478931,recycling,NaT,Private,None,container,None,None,None,...,None,None,None,None,None,None,POINT (13.42508 52.54388),"[cans, paper, cardboard, plastic, plastic_bott...",[clothes],<NA>


In [287]:
gdf[gdf["recycling:batteries"] == "no"]

,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
475,node,1256942896,recycling,2024-02-21,Deutsches Rotes Kreuz,None,container,Deutsches Rotes Kreuz,Q694104,None,...,None,None,None,None,None,None,POINT (13.18805 52.42094),"[clothes, shoes]","[glass, batteries, cans, paper, scrap_metal]",<NA>
684,node,1965564034,recycling,NaT,Deutsches Rotes Kreuz,None,container,Deutsches Rotes Kreuz,Q694104,None,...,None,None,None,None,None,None,POINT (13.5999 52.44161),"[clothes, shoes]","[glass, batteries, cans, paper, scrap_metal]",<NA>
1019,node,5792096602,recycling,2024-05-30,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.37481 52.46859),[glass_bottles],"[clothes, batteries, cans, paper, scrap_metal,...",<NA>
2824,way,1421116477,recycling,NaT,None,yes,container,None,None,None,...,None,None,None,None,None,None,"POLYGON ((13.55553 52.52773, 13.5555 52.52773,...",[glass_bottles],"[glass, batteries]",<NA>


In [288]:
gdf[gdf["not_accepted_recycling_items"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

,element,id,amenity,check_date:recycling,operator,recycling:glass_bottles,recycling_type,brand,brand:wikidata,brand:wikipedia,...,roof:colour,roof:material,roof:shape,disused:parking,recycling:organic,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
12,node,290659186,recycling,2025-10-31,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.47605 52.61573),[glass_bottles],[glass],<NA>
302,node,737216262,recycling,2023-04-16,Karl Meyer Rohstoffverwertung GmbH,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.42943 52.54495),[glass_bottles],"[clothes, cans, paper, plastic]",<NA>
475,node,1256942896,recycling,2024-02-21,Deutsches Rotes Kreuz,None,container,Deutsches Rotes Kreuz,Q694104,None,...,None,None,None,None,None,None,POINT (13.18805 52.42094),"[clothes, shoes]","[glass, batteries, cans, paper, scrap_metal]",<NA>
487,node,1299614073,recycling,NaT,None,None,container,None,None,None,...,None,None,None,None,None,None,POINT (13.5312 52.42836),"[cans, paper, plastic_packaging]",[plastic],<NA>
586,node,1576124803,recycling,2025-10-15,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.42407 52.56763),[glass_bottles],[glass],<NA>
684,node,1965564034,recycling,NaT,Deutsches Rotes Kreuz,None,container,Deutsches Rotes Kreuz,Q694104,None,...,None,None,None,None,None,None,POINT (13.5999 52.44161),"[clothes, shoes]","[glass, batteries, cans, paper, scrap_metal]",<NA>
703,node,2148931649,recycling,2025-04-05,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.47752 52.52305),[glass_bottles],[glass],<NA>
712,node,2299861699,recycling,2024-07-23,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.34159 52.53427),[glass_bottles],[glass],<NA>
718,node,2385420167,recycling,2024-12-31,None,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.58342 52.45546),[glass_bottles],[glass],<NA>
737,node,2512623620,recycling,NaT,Karl Meyer Rohstoffverwertung GmbH,yes,container,None,None,None,...,None,None,None,None,None,None,POINT (13.31065 52.58451),"[glass_bottles, glass_jars, glass_bottles_unkn...","[cans, aluminium]",<NA>


### Analyze the cleaned dataset

In [289]:
gdf_clean[gdf_clean["id"]==26867409]

,element,id,amenity,check_date:recycling,operator,recycling_type,brand,brand:wikidata,brand:wikipedia,check_date,...,building:material,roof:colour,roof:material,roof:shape,disused:parking,natural,geometry,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
0,node,26867409,recycling,2024-07-05,Berlin Recycling,container,None,None,None,None,...,None,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,<NA>


In [290]:
for col in gdf_clean.columns:
    print(col)

element
id
amenity
check_date:recycling
operator
recycling_type
brand
brand:wikidata
brand:wikipedia
check_date
description
note
location
mapillary
survey:date
source
access
ref
wheelchair
created_by
opening_hours
operator:wikidata
name
addr:city
addr:country
addr:street
addr:housenumber
addr:postcode
addr:suburb
website
check_date:opening_hours
addr:floor
covered
indoor
level
material
count
operator:phone
contact:email
contact:phone
colour
operator:short
contact:website
obstacle:parking
source:amenity
short_name
position
operator:website
green
owner
alt_name
operator:signed
phone
access:conditional
support
brand:short
email
operator:wikipedia
lit
waste
contact:mobile
fixme
contact:fax
capacity
operator:unsigned
operator:type
name:de
man_made
status
collection_times
landuse
barrier
addr:inclusion
building
fence_type
height
name:signed
industrial
building:colour
building:material
roof:colour
roof:material
roof:shape
disused:parking
natural
geometry
accepted_recycling_items
not_accepted_

In [291]:
gdf_clean[["element","id","recycling_type", "accepted_recycling_items", "not_accepted_recycling_items", "other_recycling_items"]].head()

,element,id,recycling_type,accepted_recycling_items,not_accepted_recycling_items,other_recycling_items
0,node,26867409,container,[glass_bottles],<NA>,<NA>
1,node,254985049,container,[clothes],<NA>,<NA>
2,node,262212828,container,[glass_bottles],<NA>,<NA>
3,node,267093410,container,[glass_bottles],<NA>,<NA>
4,node,272619379,container,[glass_bottles],<NA>,<NA>


In [292]:
df.geometry.iloc[0]
type(df.geometry.iloc[0])
df.geometry.geom_type.unique()

array(['Point', 'Polygon'], dtype=object)

In [293]:
help(ox.features.features_from_place)
help(ox.features)


Help on function features_from_place in module osmnx.features:

features_from_place(query: 'str | dict[str, str] | list[str | dict[str, str]]', tags: 'dict[str, bool | str | list[str]]', *, which_result: 'int | None | list[int | None]' = None) -> 'gpd.GeoDataFrame'
    Download OSM features within the boundaries of some place(s).
    
    The query must be geocodable and OSM must have polygon boundaries for the
    geocode result. If OSM does not have a polygon for this place, you can
    instead get features within it using the `features_from_address`
    function, which geocodes the place name to a point and gets the features
    within some distance of that point.
    
    If OSM does have polygon boundaries for this place but you're not finding
    it, try to vary the query string, pass in a structured query dict, or vary
    the `which_result` argument to use a different geocode result. If you know
    the OSM ID of the place, you can retrieve its boundary polygon using the
    `g

In [294]:
df.iloc[0].to_frame()

,0
element,node
id,26867409
amenity,recycling
check_date:recycling,2024-07-05 00:00:00
operator,Berlin Recycling
...,...
roof:shape,None
disused:parking,None
recycling:organic,None
natural,None


## Column Grouping Strategy

OSM metadata is highly heterogeneous.
To clean systematically, columns are grouped by semantic purpose
(e.g. identity, address, contact, accessibility).

Each group is:
- analyzed independently
- normalized using group-specific rules
- collapsed into user-facing fields


In [295]:
# Define groups of columns for analysis

groups = {
    1: ["element", "id"],
    2: ["geometry"],
    3: ["name", "operator", "operator:short", "short_name", "owner", "alt_name",
        "name:de", "operator:type", "man_made"],
    4: ["addr:street", "addr:housenumber", "addr:postcode", "addr:suburb",
        "description", "ref", "position"],
    5: ["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email"],
    6: ["website", "contact:website", "operator:website"],
    7: ["not_accepted_recycling_items", "accepted_recycling_items", "recycling_type",
        "material", "colour", "green", "waste"],
    8: ["access", "wheelchair", "obstacle:parking", "lit", "addr:floor", "level",
        "indoor", "covered", "count", "barrier"],
    9: ["opening_hours", "collection_times", "access:conditional"],
    10: ["source", "note", "fixme", "status", "landuse"],
}

In [296]:
# Filter df to include only columns that belong to at least one group
all_group_cols = [col for cols in groups.values() for col in cols]


In [297]:
# Select only relevant columns
df_selected_clean = gdf_clean[ [col for col in gdf_clean.columns if col in all_group_cols] ]
df_selected_clean.head()

,element,id,operator,recycling_type,description,note,source,access,ref,wheelchair,...,operator:type,name:de,man_made,status,collection_times,landuse,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items
0,node,26867409,Berlin Recycling,container,None,None,None,None,None,None,...,None,None,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>
1,node,254985049,None,container,2 Container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,...,None,None,None,None,None,None,None,POINT (13.32054 52.484),[clothes],<NA>
2,node,262212828,None,container,None,None,None,None,None,None,...,None,None,None,None,None,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>
3,node,267093410,None,container,None,None,None,None,None,None,...,None,None,None,None,None,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>
4,node,272619379,None,container,None,None,survey,None,None,None,...,None,None,None,None,None,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>


In [298]:
def analyze_group(df, cols):
    """
    Exploratory analysis helper.

    Purpose:
    - Inspect sparsity and overlap in related columns
    - Identify conflicting or redundant tags
    - Inform cleaning decisions downstream

    Not used in final dataset generation.
    """
    subset = df[cols]
    summary = {}

    # --- Missing values per column ---
    summary["missing_per_column"] = subset.isna().sum().to_dict()

    # --- Unique values per column ---
    summary["unique_values_per_column"] = {
        c: subset[c].dropna().unique().tolist() for c in cols
    }

    # --- Identify rows with >1 filled value ---
    # Define what counts as "filled"
    def is_filled(v):
        if pd.isna(v):
            return False
        if isinstance(v, list) and len(v) == 0:
            return False
        if isinstance(v, str) and v.strip() == "":
            return False
        return True

    filled_mask = subset.applymap(is_filled)
    filled_count = filled_mask.sum(axis=1)

    # Correct: rows where more than one column has a filled value
    summary["rows_with_multiple_filled"] = subset[filled_count > 1]

    return summary

### Group 2: Position

In [299]:
position_cols = ["geometry"]

In [300]:
df_selected_clean[df_selected_clean["geometry"].notna()][
 ["id"] + position_cols]

,id,geometry
0,26867409,POINT (13.29683 52.50133)
1,254985049,POINT (13.32054 52.484)
2,262212828,POINT (13.48568 52.51902)
3,267093410,POINT (13.59289 52.50846)
4,272619379,POINT (13.45563 52.52183)
...,...,...
2837,1448156166,"POLYGON ((13.73239 52.42862, 13.73245 52.42862..."
2838,1448681307,"POLYGON ((13.65813 52.40734, 13.65816 52.40736..."
2839,1451298080,"POLYGON ((13.59266 52.42932, 13.59269 52.42933..."
2840,1451319774,"POLYGON ((13.59724 52.43433, 13.59732 52.43437..."


In [301]:
# Ensure the GeoDataFrame uses WGS84 (longitude, latitude)
# This is required so x = longitude and y = latitude are correct
df_selected_clean = df_selected_clean.set_crs(epsg=4326, allow_override=True)


In [302]:
# Create empty latitude and longitude columns
# These will be filled for both POINT and POLYGON geometries
df_selected_clean["longitude"] = None
df_selected_clean["latitude"] = None

In [303]:
# Handle POINT geometries
# For POINT objects, longitude and latitude can be read directly
point_mask = df_selected_clean.geometry.type == "Point"

df_selected_clean.loc[point_mask, "longitude"] = df_selected_clean.loc[point_mask, "geometry"].x
df_selected_clean.loc[point_mask, "latitude"] = df_selected_clean.loc[point_mask, "geometry"].y



In [304]:
# Handle POLYGON geometries
# For POLYGON objects, compute the centroid (geometric center)
# This represents the average spatial location of the polygon
polygon_mask = df_selected_clean.geometry.type == "Polygon"

# Calculate centroids for polygon geometries
df_selected_clean.loc[polygon_mask, "centroid"] = (
    df_selected_clean.loc[polygon_mask, "geometry"].centroid
)

# Extract longitude and latitude from the centroid points
df_selected_clean.loc[polygon_mask, "longitude"] = (
    df_selected_clean.loc[polygon_mask, "centroid"].x
)
df_selected_clean.loc[polygon_mask, "latitude"] = (
    df_selected_clean.loc[polygon_mask, "centroid"].y
)

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/877852173.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_selected_clean.loc[polygon_mask, "geometry"].centroid


In [305]:
# Clean up temporary centroid column
df_selected_clean = df_selected_clean.drop(columns=["centroid"])


In [306]:
df_selected_clean.head()

,element,id,operator,recycling_type,description,note,source,access,ref,wheelchair,...,man_made,status,collection_times,landuse,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude
0,node,26867409,Berlin Recycling,container,None,None,None,None,None,None,...,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329
1,node,254985049,None,container,2 Container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,...,None,None,None,None,None,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997
2,node,262212828,None,container,None,None,None,None,None,None,...,None,None,None,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021
3,node,267093410,None,container,None,None,None,None,None,None,...,None,None,None,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457
4,node,272619379,None,container,None,None,survey,None,None,None,...,None,None,None,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829


In [307]:
df_selected_clean[df_selected_clean["longitude"].isna()][["id", "latitude", "longitude"]]


,id,latitude,longitude


### Group 3: Operator Name

#### Exploratory inspection (not part of final pipeline)


In [308]:
group3_report = analyze_group(df_selected_clean, groups[3])
group3_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'name': 2779,
  'operator': 2002,
  'operator:short': 2818,
  'short_name': 2835,
  'owner': 2838,
  'alt_name': 2836,
  'name:de': 2841,
  'operator:type': 2835,
  'man_made': 2831},
 'unique_values_per_column': {'name': ['Recyclinghof Ilsenburger Straße',
   'Altglas',
   'Bernd Klebs Recycling',
   'Fritz Pennecke Söhne Abfallentsorgung u. Recycling GmbH & Co. KG',
   'Recycling',
   'Wertstoff Ankauf Lankwitz',
   'Recon-t',
   'Beller Schrotthandel',
   'Reinhardt Rohstoffe',
   'Altkleider',
   'Recyclinghof Fischerstraße',
   'Glascontainer',
   'Batteriesammelstelle',
   'Kleiderspende',
   'Platane19',
   'Textil & Schuhe Sammelbox',
   'Wurmkiste',
   'Anti-Kompost',
   'ruhender Anti-Kompost',
   'Pfandstation',
   'Recyclinghof Berliner Straße',
   'BSR Recyclinghof Asgardstraße',
   'Recyclinghof Behmstraße',
   'BSR Recyclinghof Lengeder Straße',
   'BSR Recyclinghof Hegauer Weg',
   'BRB Baustoff Recycling',
   'Berliner Stadtreinigung Brunsbüttel

In [309]:
name_cols = ["name", "operator", "operator:short", "short_name", "owner", "alt_name",
        "name:de", "operator:type", "man_made"]

In [310]:
empty_cols = []      # columns with no data
non_empty_cols = []  # columns with data

for col in name_cols:
    uniques = df_selected_clean[col].unique()
    
    if len(uniques) > 0:
        non_empty_cols.append(col)
        print(f"\n==== {col} ====")
        print(uniques)
    else:
        empty_cols.append(col)

print("\n\nColumns with NO data:")
print(empty_cols)


==== name ====
[None 'Recyclinghof Ilsenburger Straße' 'Altglas' 'Bernd Klebs Recycling'
 'Fritz Pennecke Söhne Abfallentsorgung u. Recycling GmbH & Co. KG'
 'Recycling' 'Wertstoff Ankauf Lankwitz' 'Recon-t' 'Beller Schrotthandel'
 'Reinhardt Rohstoffe' 'Altkleider' 'Recyclinghof Fischerstraße'
 'Glascontainer' 'Batteriesammelstelle' 'Kleiderspende' 'Platane19'
 'Textil & Schuhe Sammelbox' 'Wurmkiste' 'Anti-Kompost'
 'ruhender Anti-Kompost' 'Pfandstation' 'Recyclinghof Berliner Straße'
 'BSR Recyclinghof Asgardstraße' 'Recyclinghof Behmstraße'
 'BSR Recyclinghof Lengeder Straße' 'BSR Recyclinghof Hegauer Weg'
 'BRB Baustoff Recycling' 'Berliner Stadtreinigung Brunsbütteler Damm'
 'Recyclinghof Ruppiner Chaussee' 'Wasdrack Altmetalle'
 'Recyclinghof Rahnsdorfer Straße' 'Alba' 'Marcus Knospe Wertstoffhandel'
 'BSR Recyclinghof Oberspreestraße' 'Recyclinghof Nordring'
 'BTB-Recycling-Hof' 'Paletten König / Papier Fritze'
 'Recyclinghof Gradestraße' 'Kompostplatz Lübars'
 'Recyclinghof Os

In [311]:
df_selected_clean[df_selected_clean["man_made"].notna()][
 ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
2483,12528767045,Anti-Kompost,None,None,None,None,None,None,None,composting_plant
2484,12528767054,ruhender Anti-Kompost,None,None,None,None,None,None,None,composting_plant
2485,12528767063,None,None,None,None,None,None,None,None,composting_plant
2531,12749653501,None,None,None,None,None,None,None,None,composting_plant
2567,13020637760,Anti-Kompost,None,None,None,None,None,None,None,composting_plant
2790,1353882798,None,None,None,None,None,None,None,None,composting_plant
2791,1353882799,None,None,None,None,None,None,None,None,composting_plant
2792,1353882800,None,None,None,None,None,None,None,None,composting_plant
2793,1353882804,None,None,None,None,None,None,None,None,composting_plant
2794,1353882805,None,None,None,None,None,None,None,None,composting_plant


In [312]:
df_selected_clean[df_selected_clean["operator:type"].notna()][
     ["id"] + name_cols]


,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
2209,11015694305,None,None,None,None,None,None,None,private,None
2601,28198135,Recyclinghof Behmstraße,Berliner Straßenreinigung,BSR,None,None,None,None,public,None
2613,49327130,BSR Recyclinghof Hegauer Weg,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,government,None
2647,220939822,Recyclinghof Rahnsdorfer Straße,BSR,None,None,None,None,None,public,None
2727,896165233,Kompostplatz Lübars,Hartmann Ingenieure GmbH,None,None,None,None,None,private,None
2754,1298791073,Zentraler Abfallplatz,Technische Universität Berlin,None,None,None,None,None,university,None
2798,1356668405,None,None,None,None,None,None,None,private,None


In [313]:
df_selected_clean[df_selected_clean["id"]==11015694305	]

,element,id,operator,recycling_type,description,note,source,access,ref,wheelchair,...,man_made,status,collection_times,landuse,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude
2209,node,11015694305,None,container,None,None,None,None,None,None,...,None,None,None,None,None,POINT (13.52446 52.48265),[clothes],<NA>,13.524465,52.48265


In [314]:
df_selected_clean[df_selected_clean["name:de"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
2340,11717220369,Platane19,None,None,None,None,None,Platane19 Spendenannahme,None,None


In [315]:
df_selected_clean[df_selected_clean["alt_name"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
919,4162714606,Altglas,None,None,None,None,Glasmüll;Glasabfall;Glascontainer,None,None,None
974,4853308260,Altglas,LOOP,None,None,None,Glascontainer;Glasmüll;Glasabfall,None,None,None
1519,9423596370,Altglas,None,None,None,None,Glascontainer;Glasmüll;Glasabfall,None,None,None
1912,10277175053,Altglas,None,None,None,None,Glasabfall;Glasmüll;Glascontainer,None,None,None
2108,10744741793,Altglas,None,None,None,None,Glascontainer;Glasmüll;Glasabfall,None,None,None
2110,10744886015,Altglas,None,None,None,None,Glasmüll;Glasabfall;Glascontainer,None,None,None


In [316]:
df_selected_clean[df_selected_clean["owner"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
868,3825491486,None,DRK,None,None,Deutsches Rotes Kreuz,None,None,None,None
2475,12509305495,None,Cavtex,None,None,CAVTEX,None,None,None,None
2489,12556733780,None,None,None,None,REBAT,None,None,None,None
2532,12757006046,None,None,None,None,Caritas,None,None,None,None


In [317]:
df_selected_clean[df_selected_clean["short_name"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
634,1661516003,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None
847,3726668992,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None
1091,6829879338,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None
1202,7980829652,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None
1245,8357441387,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None
1294,8511247289,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None
1410,8987596142,None,Deutsches Rotes Kreuz,None,DRK,None,None,None,None,None


In [318]:
df_selected_clean[df_selected_clean["operator:short"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
328,809926775,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
670,1892081724,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
736,2510633650,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
827,3327445141,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
1044,6053091357,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
1124,7285639747,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
1163,7634960848,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
1246,8377043001,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
1314,8574592567,None,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
1398,8909089044,None,Deutsches Rotes Kreuz,DRK,None,None,None,None,None,None


In [319]:
df_selected_clean[df_selected_clean["operator"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
0,26867409,None,Berlin Recycling,None,None,None,None,None,None,None
6,273174898,None,Rhenus,None,None,None,None,None,None,None
15,292935637,None,Karl Meyer Rohstoffverwertung,None,None,None,None,None,None,None
20,305943442,None,Rhenus,None,None,None,None,None,None,None
21,306456511,None,Karl Meyer Rohstoffverwertung,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
2758,1311349821,None,Cavuslar,None,None,None,None,None,None,None
2759,1311349822,None,Humana,None,None,None,None,None,None,None
2760,1311349823,None,Humana,None,None,None,None,None,None,None
2770,1317973378,None,Humana,None,None,None,None,None,None,None


In [320]:
df_selected_clean[df_selected_clean["name"].notna()][
     ["id"] + name_cols]

,id,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
115,442877823,Recyclinghof Ilsenburger Straße,BSR,None,None,None,None,None,None,None
169,499987314,Altglas,None,None,None,None,None,None,None,None
521,1434318202,Bernd Klebs Recycling,None,None,None,None,None,None,None,None
680,1962417491,Altglas,None,None,None,None,None,None,None,None
744,2525335317,Fritz Pennecke Söhne Abfallentsorgung u. Recyc...,REMONDIS,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
2728,966672349,Recyclinghof Ostpreußendamm,Berliner Stadtreinigungsbetriebe,BSR,None,None,None,None,None,None
2734,1051824179,EBU Dienstleistungs- GmbH,None,None,None,None,None,None,None,None
2745,1200769075,Recyclinghof Brunsbütteler Damm,Berliner Stadtreinigungsbetriebe Anstalt des ö...,None,None,None,None,None,None,None
2754,1298791073,Zentraler Abfallplatz,Technische Universität Berlin,None,None,None,None,None,university,None


#### Apply normalization (final pipeline)

In [321]:
def combine_names(row):
    """
    Combines name-related columns into:
    - display_name: a single best human-readable label (or pd.NA if none exists)
    - name_metadata: list of all name/operator-related values with provenance
    - entity_type: list of classification-related values

    Design choice:
    - We do NOT invent names from entity types (e.g. man_made).
    - Operator is only used as a fallback identifier when no name-like field exists.
    """

    # --- DISPLAY NAME ---
    display_name = pd.NA

    name_priority = [
        "name",
        "short_name",
        "name:de",
        "alt_name",
    ]

    for c in name_priority:
        val = row.get(c)
        if pd.notna(val):
            display_name = val
            break

    # Use operator only if no name exists
    if pd.isna(display_name):
        for c in ["operator", "operator:short"]:
            val = row.get(c)
            if pd.notna(val):
                display_name = val
                break

    # --- NAME METADATA ---
    metadata_cols = [
        "name",
        "short_name",
        "name:de",
        "alt_name",
        "operator",
        "operator:short",
        "owner",
    ]

    name_metadata = []
    for c in metadata_cols:
        val = row.get(c)
        if pd.notna(val):
            name_metadata.append(f"{c}: {val}")

    if not name_metadata:
        name_metadata = pd.NA

    # --- ENTITY TYPE ---
    type_cols = ["operator:type", "man_made"]

    entity_type = []
    for c in type_cols:
        val = row.get(c)
        if pd.notna(val):
            entity_type.append(f"{c}: {val}")

    if not entity_type:
        entity_type = pd.NA

    return pd.Series(
        {
            "display_name": display_name,
            "name_metadata": name_metadata,
            "entity_type": entity_type,
        }
    )


In [322]:
# Apply to dataframe
df_selected_clean = df_selected_clean.join(df_selected_clean.apply(combine_names, axis=1))
df_selected_clean.head()

,element,id,operator,recycling_type,description,note,source,access,ref,wheelchair,...,landuse,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type
0,node,26867409,Berlin Recycling,container,None,None,None,None,None,None,...,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>
1,node,254985049,None,container,2 Container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,...,None,None,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>
2,node,262212828,None,container,None,None,None,None,None,None,...,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>
3,node,267093410,None,container,None,None,None,None,None,None,...,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>
4,node,272619379,None,container,None,None,survey,None,None,None,...,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>


In [323]:
df_selected_clean[
     ["display_name", "name_metadata", "entity_type"] + name_cols]

,display_name,name_metadata,entity_type,name,operator,operator:short,short_name,owner,alt_name,name:de,operator:type,man_made
0,Berlin Recycling,[operator: Berlin Recycling],<NA>,None,Berlin Recycling,None,None,None,None,None,None,None
1,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
2,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
3,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
4,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
2837,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
2838,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
2839,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None
2840,<NA>,<NA>,<NA>,None,None,None,None,None,None,None,None,None


In [324]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=name_cols, inplace=True)
df_selected_clean.head()

,element,id,recycling_type,description,note,source,access,ref,wheelchair,opening_hours,...,landuse,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type
0,node,26867409,container,None,None,None,None,None,None,None,...,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>
1,node,254985049,container,2 Container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,...,None,None,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,None,None,...,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,None,None,...,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>
4,node,272619379,container,None,None,survey,None,None,None,None,...,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>


### Group 4: Address

#### Exploratory inspection (not part of final pipeline)

In [325]:
group4_report = analyze_group(df_selected_clean, groups[4])
group4_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'addr:street': 2811,
  'addr:housenumber': 2818,
  'addr:postcode': 2816,
  'addr:suburb': 2825,
  'description': 2821,
  'ref': 2756,
  'position': 2841},
 'unique_values_per_column': {'addr:street': ['Bochumer Straße',
   'Ilsenburger Straße',
   'Grellstraße',
   'Barnackufer',
   'Greinerstraße',
   'Lahnstraße',
   'Haynauer Straße',
   'Buchberger Straße',
   'Liebensteiner Straße',
   'Lichterfelder Weg',
   'Techowpromenade',
   'Roedernallee',
   'Am Nordgraben',
   'Pestalozzistraße',
   'Berliner Straße',
   'Asgardstraße',
   'Behmstraße',
   'Lengeder Straße',
   'Hegauer Weg',
   'Brunsbütteler Damm',
   'Ruppiner Chaussee',
   'Rahnsdorfer Straße',
   'Fischerstraße',
   'Grabensprung',
   'Nordring',
   'Gradestraße',
   'Alter Bernauer Heerweg',
   'Ostpreußendamm',
   'Straße des 17. Juni'],
  'addr:housenumber': ['18-20',
   '27',
   '31',
   '65',
   '30',
   '1',
   '60',
   '110',
   '3',
   '74',
   '6-18',
   '17',
   '43',
   '341',
   '

In [326]:
address_cols=["addr:street", "addr:housenumber", "addr:postcode", "addr:suburb",
        "description", "ref", "position"]

In [327]:
df_selected_clean[df_selected_clean["position"].notna()][
 ["id"] + address_cols]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
659,1834091939,None,None,None,None,None,None,lane


In [328]:
df_selected_clean[df_selected_clean["ref"].notna()][
 ["id"] + address_cols]


,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
22,309364526,None,None,None,None,None,Hohenzollerndamm 208 / Wasserwerk,None
23,309847736,None,None,None,None,None,Paulsborner Str. 70 ggü. Nr. 27 Ecke Seesener ...,None
25,311090440,None,None,None,None,None,Eisenzahnstr. Ecke Paulsborner Str.,None
26,311090625,None,None,None,None,None,Ballenstedter Str. 2 Ecke Brandenburgische Str.,None
28,312385354,None,None,None,None,None,Olivaer Platz ggü. Nr. 12,None
...,...,...,...,...,...,...,...,...
2679,372635362,None,None,None,None,None,K,None
2689,464446146,None,None,None,None,None,J,None
2690,464446162,None,None,None,None,None,L,None
2691,464446166,None,None,None,None,None,M,None


In [329]:
df_selected_clean[df_selected_clean["description"].notna()][
 ["id"] + address_cols]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
1,254985049,None,None,None,None,2 Container,None,None
115,442877823,Ilsenburger Straße,18-20,10589,Charlottenburg,Berliner Stadtreinigungsbetriebe,None,None
228,663611902,None,None,None,None,2 Container,None,None
291,718143788,None,None,None,None,insgesamt 5 Container,Olympische Str. 30 - 34,None
318,783500754,None,None,None,None,Altglascontainer,None,None
470,1240609775,None,None,None,None,Fritz-Wildung-Str. Ecke Cunostr. im Parkhafen,None,None
678,1933518071,None,None,None,None,Glas Recycling,None,None
744,2525335317,Greinerstraße,27,12107,Mariendorf,"Entsorgungsunternehmen, Containerdienst, FPS",None,None
984,4972103512,None,None,None,None,"Altglas-Container weiß, grün, braun",None,None
988,5167671611,None,None,None,None,2 Container,None,None


In [330]:
df_selected_clean[df_selected_clean["addr:suburb"].notna()][
 ["id"] + address_cols]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
115,442877823,Ilsenburger Straße,18-20,10589,Charlottenburg,Berliner Stadtreinigungsbetriebe,None,None
521,1434318202,Barnackufer,27,12207,Lichterfelde,None,None,None
744,2525335317,Greinerstraße,27,12107,Mariendorf,"Entsorgungsunternehmen, Containerdienst, FPS",None,None
773,2820307602,Lahnstraße,31,12055,Neukölln,None,None,None
823,3309835874,Haynauer Straße,65,12249,Lankwitz,None,None,None
1196,7951636690,Lichterfelder Weg,1,14167,Lichterfelde,None,None,None
2600,26660877,Asgardstraße,3,13089,Heinersdorf,None,None,None
2601,28198135,Behmstraße,74,10439,Prenzlauer Berg,None,None,None
2602,28760600,Lengeder Straße,6-18,13407,Reinickendorf,None,None,None
2620,104677645,Brunsbütteler Damm,43,13581,Spandau,None,None,None


In [331]:
df_selected_clean[df_selected_clean["addr:postcode"].notna()][
 ["id"] + address_cols]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
115,442877823,Ilsenburger Straße,18-20,10589,Charlottenburg,Berliner Stadtreinigungsbetriebe,None,None
302,737216262,Grellstraße,None,10409,None,None,None,None
521,1434318202,Barnackufer,27,12207,Lichterfelde,None,None,None
744,2525335317,Greinerstraße,27,12107,Mariendorf,"Entsorgungsunternehmen, Containerdienst, FPS",None,None
773,2820307602,Lahnstraße,31,12055,Neukölln,None,None,None
823,3309835874,Haynauer Straße,65,12249,Lankwitz,None,None,None
851,3727940016,Buchberger Straße,30,10365,None,None,None,None
946,4489294193,Liebensteiner Straße,None,12687,None,None,None,None
947,4489294195,Liebensteiner Straße,None,12687,None,None,None,None
1196,7951636690,Lichterfelder Weg,1,14167,Lichterfelde,None,None,None


In [332]:
df_selected_clean[df_selected_clean["addr:housenumber"].notna()][
 ["id"] + address_cols]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
115,442877823,Ilsenburger Straße,18-20,10589,Charlottenburg,Berliner Stadtreinigungsbetriebe,None,None
521,1434318202,Barnackufer,27,12207,Lichterfelde,None,None,None
744,2525335317,Greinerstraße,27,12107,Mariendorf,"Entsorgungsunternehmen, Containerdienst, FPS",None,None
773,2820307602,Lahnstraße,31,12055,Neukölln,None,None,None
823,3309835874,Haynauer Straße,65,12249,Lankwitz,None,None,None
851,3727940016,Buchberger Straße,30,10365,None,None,None,None
1196,7951636690,Lichterfelder Weg,1,14167,Lichterfelde,None,None,None
2340,11717220369,Pestalozzistraße,60,10627,None,None,None,None
2599,10734959,Berliner Straße,110,10713,None,None,None,None
2600,26660877,Asgardstraße,3,13089,Heinersdorf,None,None,None


In [333]:
df_selected_clean[df_selected_clean["addr:street"].notna()][
 ["id"] + address_cols]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position
86,393769325,Bochumer Straße,None,None,None,None,None,None
115,442877823,Ilsenburger Straße,18-20,10589,Charlottenburg,Berliner Stadtreinigungsbetriebe,None,None
302,737216262,Grellstraße,None,10409,None,None,None,None
521,1434318202,Barnackufer,27,12207,Lichterfelde,None,None,None
744,2525335317,Greinerstraße,27,12107,Mariendorf,"Entsorgungsunternehmen, Containerdienst, FPS",None,None
773,2820307602,Lahnstraße,31,12055,Neukölln,None,None,None
823,3309835874,Haynauer Straße,65,12249,Lankwitz,None,None,None
851,3727940016,Buchberger Straße,30,10365,None,None,None,None
946,4489294193,Liebensteiner Straße,None,12687,None,None,None,None
947,4489294195,Liebensteiner Straße,None,12687,None,None,None,None


#### Apply normalization (final pipeline)

In [334]:
def build_structured_address(row):
    """
    Build a human-readable street address from structured OSM addr:* tags.

    Rationale:
    - addr:street, addr:housenumber, addr:postcode, addr:suburb
      are the highest-quality, machine-curated fields.
    - We only use these fields here to avoid contaminating
      clean addresses with free-text noise.
    """

    parts = []

    # --- Street + house number ---
    # Street is mandatory for a valid structured address.
    # House number is optional and appended only if present.
    if pd.notna(row["addr:street"]):
        street_part = row["addr:street"]

        if pd.notna(row["addr:housenumber"]):
            street_part += f" {row['addr:housenumber']}"

        parts.append(street_part)

    # --- Postcode + suburb ---
    # These are grouped together as a "city part".
    # Either one may exist independently.
    city_part = []

    if pd.notna(row["addr:postcode"]):
        city_part.append(str(row["addr:postcode"]))

    if pd.notna(row["addr:suburb"]):
        city_part.append(row["addr:suburb"])

    if city_part:
        parts.append(" ".join(city_part))

    # If no structured components were available, return NA
    # so that fallback logic can take over explicitly.
    return ", ".join(parts) if parts else pd.NA

In [335]:
# Apply row-wise because we are combining multiple columns conditionally
df_selected_clean["full_address"] = df_selected_clean.apply(build_structured_address, axis=1)
df_selected_clean.head()


,element,id,recycling_type,description,note,source,access,ref,wheelchair,opening_hours,...,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address
0,node,26867409,container,None,None,None,None,None,None,None,...,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,<NA>
1,node,254985049,container,2 Container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,...,None,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,None,None,...,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,None,None,...,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,<NA>
4,node,272619379,container,None,None,survey,None,None,None,None,...,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,<NA>


In [336]:
df_selected_clean[
 ["id"] + address_cols + ["full_address"]]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position,full_address
0,26867409,None,None,None,None,None,None,None,<NA>
1,254985049,None,None,None,None,2 Container,None,None,<NA>
2,262212828,None,None,None,None,None,None,None,<NA>
3,267093410,None,None,None,None,None,None,None,<NA>
4,272619379,None,None,None,None,None,None,None,<NA>
...,...,...,...,...,...,...,...,...,...
2837,1448156166,None,None,None,None,None,None,None,<NA>
2838,1448681307,None,None,None,None,None,None,None,<NA>
2839,1451298080,None,None,None,None,None,None,None,<NA>
2840,1451319774,None,None,None,None,None,None,None,<NA>


In [337]:
df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + address_cols + ["full_address"]]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position,full_address
86,393769325,Bochumer Straße,None,None,None,None,None,None,Bochumer Straße
115,442877823,Ilsenburger Straße,18-20,10589,Charlottenburg,Berliner Stadtreinigungsbetriebe,None,None,"Ilsenburger Straße 18-20, 10589 Charlottenburg"
302,737216262,Grellstraße,None,10409,None,None,None,None,"Grellstraße, 10409"
521,1434318202,Barnackufer,27,12207,Lichterfelde,None,None,None,"Barnackufer 27, 12207 Lichterfelde"
744,2525335317,Greinerstraße,27,12107,Mariendorf,"Entsorgungsunternehmen, Containerdienst, FPS",None,None,"Greinerstraße 27, 12107 Mariendorf"
773,2820307602,Lahnstraße,31,12055,Neukölln,None,None,None,"Lahnstraße 31, 12055 Neukölln"
823,3309835874,Haynauer Straße,65,12249,Lankwitz,None,None,None,"Haynauer Straße 65, 12249 Lankwitz"
851,3727940016,Buchberger Straße,30,10365,None,None,None,None,"Buchberger Straße 30, 10365"
946,4489294193,Liebensteiner Straße,None,12687,None,None,None,None,"Liebensteiner Straße, 12687"
947,4489294195,Liebensteiner Straße,None,12687,None,None,None,None,"Liebensteiner Straße, 12687"


Address Fallback Logic

Not all OSM entries contain structured `addr:*` tags.
When structured addresses are missing, we fall back to free-text fields
in a controlled priority order, explicitly trading precision for coverage.


In [338]:
# Backup sources ordered by decreasing reliability
# ref        → often contains a usable street + number
# description→ sometimes contains address-like text
# position   is not used due to low quality, intstead reverse search is applied
fallback_cols = ["ref", "description"]

Filter fallback values with simple heuristics.
We only want strings that look like addresses, e.g.:
- contain a street suffix (str, straße, platz, damm, etc.)
- contain a number
- or contain keywords like Ecke, ggü.

In [339]:
ADDRESS_PATTERN = re.compile(
    r"""
    (
        # Street names
        \b(str\.|straße|strasse|platz|allee|damm|weg|ufer|ring)\b
        |
        # Intersection / relative location
        \b(ecke|ggü\.?|gegenüber)\b
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

def looks_like_address(value):
    if pd.isna(value):
        return False
    value = str(value).strip()
    if len(value) < 5:
        return False
    return bool(ADDRESS_PATTERN.search(value))


In [340]:
# Apply fallback ONLY if it passes validation
fallback_series = (
    df_selected_clean[["ref", "description"]]
    .apply(
        lambda row: next(
            (
                val for val in row
                if looks_like_address(val)
            ),
            pd.NA
        ),
        axis=1
    )
    .astype("string")
)

df_selected_clean["full_address"] = (
    df_selected_clean["full_address"]
    .astype("string")
    .fillna(fallback_series)
)


In [341]:
print(df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + address_cols + ["full_address"]])

              id             addr:street addr:housenumber addr:postcode  \
23     309847736                    None             None          None   
25     311090440                    None             None          None   
26     311090625                    None             None          None   
28     312385354                    None             None          None   
78     374420687                    None             None          None   
...          ...                     ...              ...           ...   
2704   579730522             Gradestraße               73         12347   
2727   896165233  Alter Bernauer Heerweg               44         13469   
2728   966672349          Ostpreußendamm                1         12207   
2745  1200769075      Brunsbütteler Damm               47         13581   
2754  1298791073     Straße des 17. Juni              144          None   

       addr:suburb description  \
23            None        None   
25            None        None 

In [342]:
print(df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + ["full_address"]])

              id                                       full_address
23     309847736  Paulsborner Str. 70 ggü. Nr. 27 Ecke Seesener ...
25     311090440                Eisenzahnstr. Ecke Paulsborner Str.
26     311090625    Ballenstedter Str. 2 Ecke Brandenburgische Str.
28     312385354                          Olivaer Platz ggü. Nr. 12
78     374420687                    Leibnizstr. 65 Ecke Niebuhrstr.
...          ...                                                ...
2704   579730522                        Gradestraße 73, 12347 Britz
2727   896165233            Alter Bernauer Heerweg 44, 13469 Lübars
2728   966672349               Ostpreußendamm 1, 12207 Lichterfelde
2745  1200769075               Brunsbütteler Damm 47, 13581 Spandau
2754  1298791073                            Straße des 17. Juni 144

[89 rows x 2 columns]


Reverse search position for address

In [ ]:
"""# Initialize a new Nominatim geocoder instance
geolocator = Nominatim(user_agent="berlin_recycling_locator")
"""

In [ ]:
""'''def reverse_geocode_address(lat, lon):
    """
    Reverse geocode coordinates into a clean, human-readable address.

    Used ONLY as a last-resort fallback when structured OSM data is missing.
    """


    print("CALL:", lat, lon)
  

    try:
        print("TRY")
        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True,
            language="de"
        )
        print("location:", location)
        sleep(1)  # Respect Nominatim rate limits

        if not location or "address" not in location.raw:
            print("NO LOCATION")
            return pd.NA

        addr = location.raw["address"]
        print("ADDR:", addr)

        parts = []

        # Street + house number
        if addr.get("road"):
            print("HAS ROAD")
            street = addr["road"]

            if addr.get("house_number"):

                print("HAS HOUSE NUMBER")
                street += f" {addr['house_number']}"
            parts.append(street)

        # Postcode + city
        city_part = []
        if addr.get("postcode"):
            print("HAS POSTCODE")
            city_part.append(addr["postcode"])
        if addr.get("city") or addr.get("town"):
            print("HAS CITY/TOWN")
            city_part.append(addr.get("city") or addr.get("town"))

        if city_part:
            print("HAS CITY PART")
            parts.append(" ".join(city_part))

        return ", ".join(parts) if parts else pd.NA

    except Exception:
        return pd.NA'''""


In [345]:
df_selected_clean["full_address"].describe()

count                              89
unique                             88
top       Liebensteiner Straße, 12687
freq                                2
Name: full_address, dtype: object

In [346]:
# Apply only where full_address is missing

missing_address_mask = (
    df_selected_clean["full_address"].isna() &
    df_selected_clean["latitude"].notna() &
    df_selected_clean["longitude"].notna()
)




In [347]:
missing_address_mask.info()

<class 'pandas.core.series.Series'>
RangeIndex: 2842 entries, 0 to 2841
Series name: None
Non-Null Count  Dtype
--------------  -----
2842 non-null   bool 
dtypes: bool(1)
memory usage: 2.9 KB


In [ ]:
"""df_selected_clean.loc[missing_address_mask, "full_address"] = (
    df_selected_clean.loc[missing_address_mask]
    .apply(
        lambda row: reverse_geocode_address(row["latitude"], row["longitude"]),
        axis=1
    )
)"""

In [456]:
df_selected_clean[df_selected_clean["full_address"].isna()]

,element,id,source,landuse,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,...,access_restriction,wheelchair_access,physical_obstacles,environmental_features,floor_level,unit_count,accessibility_features,availability_info,is_operational,join_geometry
0,node,26867409,<NA>,<NA>,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.29683 52.50133)
1,node,254985049,<NA>,<NA>,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.32054 52.484)
2,node,262212828,<NA>,<NA>,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.48568 52.51902)
3,node,267093410,<NA>,<NA>,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.59289 52.50846)
4,node,272619379,survey,<NA>,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.45563 52.52183)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2837,way,1448156166,<NA>,<NA>,"POLYGON ((13.73239 52.42862, 13.73245 52.42862...",<NA>,<NA>,13.732412,52.428596,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.73241 52.4286)
2838,way,1448681307,<NA>,<NA>,"POLYGON ((13.65813 52.40734, 13.65816 52.40736...","[glass_bottles, clothes]",[glass],13.658179,52.407331,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.65818 52.40733)
2839,way,1451298080,<NA>,<NA>,"POLYGON ((13.59266 52.42932, 13.59269 52.42933...",[glass_bottles],<NA>,13.592682,52.429286,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.59268 52.42929)
2840,way,1451319774,<NA>,<NA>,"POLYGON ((13.59724 52.43433, 13.59732 52.43437...",[glass_bottles],<NA>,13.597282,52.434349,<NA>,...,<NA>,<NA>,parking obstruction,<NA>,<NA>,<NA>,Obstacles: parking obstruction,<NA>,<NA>,POINT (13.59728 52.43435)


In [457]:
missing_test = df_selected_clean[df_selected_clean["id"].isin([26867409,254985049,262212828,1451319774,1452164030])]

In [460]:
missing_test_flag = df_selected_clean["id"].isin([26867409,254985049,262212828,1451319774,1452164030])
missing_test_flag


0        True
1        True
2        True
3       False
4       False
        ...  
2837    False
2838    False
2839    False
2840     True
2841     True
Name: id, Length: 2842, dtype: bool

In [477]:
# Initialize a new Nominatim geocoder instance
geolocator = Nominatim(user_agent="berlin_recycling_locator")

In [478]:
def reverse_geocode_address(lat, lon):
    try:
        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True,
            language="de"
        )
        sleep(1)

        if not location:
            return pd.NA

        raw = location.raw
        addr = raw.get("address", {})

        parts = []

        # Street + house number
        if addr.get("road"):
            street = addr["road"]
            if addr.get("house_number"):
                street += f" {addr['house_number']}"
            parts.append(street)

        # Postcode + city
        city_part = []
        if addr.get("postcode"):
            city_part.append(addr["postcode"])
        if addr.get("city") or addr.get("town"):
            city_part.append(addr.get("city") or addr.get("town"))

        if city_part:
            parts.append(" ".join(city_part))

        # ✅ Structured address available
        if parts:
            return ", ".join(parts)

        # 🔑 Fallback: human-readable name
        if raw.get("display_name"):
            return raw["display_name"]

        return pd.NA

    except Exception as e:
        print("ERROR:", e)
        return pd.NA


In [480]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="test_ssl")

location = geolocator.reverse((52.5013289, 13.296828), language="de")
print(location)


GeocoderUnavailable: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=52.5013289&lon=13.296828&format=json&accept-language=de&addressdetails=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)')))

In [479]:
df_test=df_selected_clean
df_test.loc[missing_test_flag, "full_address"] = (
    df_test.loc[missing_test_flag]
    .apply(
        lambda row: reverse_geocode_address(row["latitude"], row["longitude"]),
        axis=1
    )
)

ERROR: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=52.5013289&lon=13.296828&format=json&accept-language=de&addressdetails=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)')))
ERROR: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=52.4839968&lon=13.3205366&format=json&accept-language=de&addressdetails=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)')))
ERROR: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /reverse?lat=52.5190209&lon=13.4856806&format=json&accept-language=de&addressdetails=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate v

In [465]:
df_test[df_test["id"].isin([26867409,254985049,262212828,1451319774,1452164030])][["id","full_address"]]

,id,full_address
0,26867409,<NA>
1,254985049,<NA>
2,262212828,<NA>
2840,1451319774,<NA>
2841,1452164030,<NA>


In [349]:
print(
    "Addresses filled via Nominatim:",
    missing_address_mask.sum()
)


Addresses filled via Nominatim: 2753


In [455]:
df_selected_clean["full_address"].isna().sum()


np.int64(2753)

In [350]:
df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + address_cols + ["full_address"]]

,id,addr:street,addr:housenumber,addr:postcode,addr:suburb,description,ref,position,full_address
23,309847736,None,None,None,None,None,Paulsborner Str. 70 ggü. Nr. 27 Ecke Seesener ...,None,Paulsborner Str. 70 ggü. Nr. 27 Ecke Seesener ...
25,311090440,None,None,None,None,None,Eisenzahnstr. Ecke Paulsborner Str.,None,Eisenzahnstr. Ecke Paulsborner Str.
26,311090625,None,None,None,None,None,Ballenstedter Str. 2 Ecke Brandenburgische Str.,None,Ballenstedter Str. 2 Ecke Brandenburgische Str.
28,312385354,None,None,None,None,None,Olivaer Platz ggü. Nr. 12,None,Olivaer Platz ggü. Nr. 12
78,374420687,None,None,None,None,None,Leibnizstr. 65 Ecke Niebuhrstr.,None,Leibnizstr. 65 Ecke Niebuhrstr.
...,...,...,...,...,...,...,...,...,...
2704,579730522,Gradestraße,73,12347,Britz,None,None,None,"Gradestraße 73, 12347 Britz"
2727,896165233,Alter Bernauer Heerweg,44,13469,Lübars,None,None,None,"Alter Bernauer Heerweg 44, 13469 Lübars"
2728,966672349,Ostpreußendamm,1,12207,Lichterfelde,None,None,None,"Ostpreußendamm 1, 12207 Lichterfelde"
2745,1200769075,Brunsbütteler Damm,47,13581,Spandau,None,None,None,"Brunsbütteler Damm 47, 13581 Spandau"


In [351]:
df_selected_clean["full_address"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2842 entries, 0 to 2841
Series name: full_address
Non-Null Count  Dtype 
--------------  ----- 
89 non-null     string
dtypes: string(1)
memory usage: 22.3 KB


In [352]:
df_selected_clean["full_address"].describe()

count                              89
unique                             88
top       Liebensteiner Straße, 12687
freq                                2
Name: full_address, dtype: object

In [353]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=address_cols, inplace=True)
df_selected_clean.head()

,element,id,recycling_type,note,source,access,wheelchair,opening_hours,website,addr:floor,...,barrier,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address
0,node,26867409,container,None,None,None,None,None,None,None,...,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,<NA>
1,node,254985049,container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,None,...,None,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,None,None,...,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,None,None,...,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,<NA>
4,node,272619379,container,None,survey,None,None,None,None,None,...,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,<NA>


### Group 5: Contact

#### Exploratory inspection (not part of final pipeline)

In [354]:
group5_report = analyze_group(df_selected_clean, groups[5])
group5_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'operator:phone': 2840,
  'contact:phone': 2825,
  'phone': 2830,
  'contact:mobile': 2839,
  'contact:fax': 2840,
  'contact:email': 2831},
 'unique_values_per_column': {'operator:phone': ['+49 30 29666893',
   '+498005889934'],
  'contact:phone': ['+49 163 954 77 95',
   '033425087777',
   '+49 157 56233471',
   '+49 30 69033-535',
   '+49 157 33566415',
   '+49 30 20 04 68 73',
   '+49 30 5099-679',
   '+491625857037',
   '+49 800 5889934',
   '+49 157 56 23 34 71',
   '+49 1575 6233471',
   '+49 30 75924900',
   '+49 30 40203240',
   '+49 30 513009314'],
  'phone': ['+49 8003344140',
   '+49 151 47418192',
   '+49 163 954 77 95',
   '+493075924900',
   '+491784687106',
   '+4915755605712',
   '+49 30 30307752',
   '015733566415',
   '+49 30 3955847',
   '+49 30 75924900'],
  'contact:mobile': ['+49 157 56233471',
   '+49 152 10142291',
   '+49 1604 56 22 56'],
  'contact:fax': ['+49 30 29666894', '+49 30 40203250'],
  'contact:email': ['servicecontainer@web.

In [355]:
contact_cols=["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email"] 

In [356]:
df_selected_clean[df_selected_clean["operator:phone"].notna()][
 ["id"] + contact_cols]

,id,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email
224,656450757,+49 30 29666893,None,None,None,None,None
737,2512623620,+498005889934,None,None,None,None,None


In [357]:
df_selected_clean[df_selected_clean["contact:phone"].notna()][
 ["id"] + contact_cols]

,id,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email
265,691484559,None,+49 163 954 77 95,None,None,None,servicecontainer@web.de
1149,7454939230,None,033425087777,None,None,None,None
1150,7455040827,None,+49 157 56233471,None,None,None,info@cavtex.de
1375,8761752958,None,+49 30 69033-535,None,None,None,None
1427,9121193636,None,+49 157 33566415,None,None,None,GAK-Textilrecycling@gmx.de
1430,9134637158,None,+49 30 20 04 68 73,None,None,None,None
1431,9134637159,None,+49 157 33566415,None,None,None,GAK-Textilrecycling@gmx.de
1432,9134936853,None,+49 157 33566415,None,None,None,GAK-Textilrecycling@gmx.de
1480,9218731332,None,+49 30 5099-679,None,None,None,None
1501,9332704129,None,+491625857037,None,None,None,None


In [358]:
df_selected_clean[df_selected_clean["phone"].notna()][
 ["id"] + contact_cols]

,id,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email
984,4972103512,None,None,+49 8003344140,None,None,None
1115,7126769776,None,None,+49 151 47418192,None,None,None
1440,9163472917,None,None,+49 163 954 77 95,None,None,None
1803,10079244117,None,None,+493075924900,None,None,None
2213,11026005421,None,None,+491784687106,None,None,None
2214,11026028994,None,None,+4915755605712,None,None,None
2340,11717220369,None,None,+49 30 30307752,None,None,None
2537,12776622193,None,None,015733566415,None,None,None
2602,28760600,None,None,+493075924900,None,None,None
2623,125208195,None,None,+49 30 3955847,None,None,None


In [359]:
df_selected_clean[df_selected_clean["contact:mobile"].notna()][
 ["id"] + contact_cols]

,id,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email
1306,8544127446,None,None,None,+49 157 56233471,None,None
1551,9549407443,None,None,None,+49 152 10142291,None,simovvalentin00@gmail.com
1565,9566349245,None,None,None,+49 1604 56 22 56,None,None


In [360]:
df_selected_clean[df_selected_clean["contact:fax"].notna()][
 ["id"] + contact_cols]

,id,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email
1552,9549407444,None,+49 800 5889934,None,None,+49 30 29666894,berlin@karl-meyer.de
2727,896165233,None,+49 30 40203240,None,None,+49 30 40203250,info@hi-g.de


In [361]:
df_selected_clean[df_selected_clean["contact:email"].notna()][
 ["id"] + contact_cols]

,id,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email
265,691484559,None,+49 163 954 77 95,None,None,None,servicecontainer@web.de
1150,7455040827,None,+49 157 56233471,None,None,None,info@cavtex.de
1427,9121193636,None,+49 157 33566415,None,None,None,GAK-Textilrecycling@gmx.de
1431,9134637159,None,+49 157 33566415,None,None,None,GAK-Textilrecycling@gmx.de
1432,9134936853,None,+49 157 33566415,None,None,None,GAK-Textilrecycling@gmx.de
1551,9549407443,None,None,None,+49 152 10142291,None,simovvalentin00@gmail.com
1552,9549407444,None,+49 800 5889934,None,None,+49 30 29666894,berlin@karl-meyer.de
1774,10040314099,None,+49 157 56 23 34 71,None,None,None,info@cavtex.de
2093,10721265342,None,+49 1575 6233471,None,None,None,info@cavtex.de
2727,896165233,None,+49 30 40203240,None,None,+49 30 40203250,info@hi-g.de


#### Apply normalization (final pipeline)

In [362]:
# Define function to combine specified columns with optional prefix
def combine_columns(row, cols, prefix=True, sep="; "):
    values = []

    for c in cols:
        val = row.get(c)
        if pd.notna(val):
            values.append(f"{c}: {val}" if prefix else str(val))

    return sep.join(values) if values else pd.NA

In [363]:
# Assign combined phones to new column
phone_cols = [
    "operator:phone",
    "contact:phone",
    "phone",
    "contact:mobile",
    "contact:fax", 
    "contact:email"
]
df_selected_clean["contact_combined"] = df_selected_clean.apply(
    combine_columns,
    axis=1,
    cols=phone_cols
)
df_selected_clean.head()

,element,id,recycling_type,note,source,access,wheelchair,opening_hours,website,addr:floor,...,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address,contact_combined
0,node,26867409,container,None,None,None,None,None,None,None,...,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,<NA>,<NA>
1,node,254985049,container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,None,...,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,None,None,...,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,None,None,...,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,container,None,survey,None,None,None,None,None,...,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,<NA>,<NA>


In [364]:
# View rows of group 5 where fax is present as a test case
df_selected_clean[df_selected_clean["contact:fax"].notna()][
    ["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email", "contact_combined"]
]


,operator:phone,contact:phone,phone,contact:mobile,contact:fax,contact:email,contact_combined
1552,None,+49 800 5889934,None,None,+49 30 29666894,berlin@karl-meyer.de,contact:phone: +49 800 5889934; contact:fax: +...
2727,None,+49 30 40203240,None,None,+49 30 40203250,info@hi-g.de,contact:phone: +49 30 40203240; contact:fax: +...


In [365]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email"], inplace=True)
df_selected_clean.head()

,element,id,recycling_type,note,source,access,wheelchair,opening_hours,website,addr:floor,...,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address,contact_combined
0,node,26867409,container,None,None,None,None,None,None,None,...,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,<NA>,<NA>
1,node,254985049,container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,None,...,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,None,None,...,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,None,None,...,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,container,None,survey,None,None,None,None,None,...,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,<NA>,<NA>


### Group 6: Website

#### Exploratory inspection (not part of final pipeline)

In [366]:
group6_report = analyze_group(df_selected_clean, groups[6])
group6_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'website': 2813,
  'contact:website': 2823,
  'operator:website': 2841},
 'unique_values_per_column': {'website': ['https://www.bsr.de/8856.html',
   'https://www.klebs.info/',
   'https://www.fps-entsorgung.de/',
   'http://www.drk-berlin-city.de',
   'https://www.berlin-recycling.de',
   'http://www.berlin-textilrecycling.com/',
   'https://www.drk.de/',
   'https://www.drk.de/kleidersammlung',
   'https://www.bsr.de/recyclinghoefe-20503.php?currRCLocation=42cdc763-e67e-4e4d-98d7-6e081ce959bd&view=list',
   'https://www.platane19.de/laeden/gebrauchtwarenlaeden/spendenannahme/',
   'https://cavtex.de',
   'https://www.berliner-stadtmission.de/sachspenden',
   'https://www.bsr.de/recyclinghoefe-20503.php?currRCLocation=&view=list',
   'https://www.bsr.de/8859.html',
   'https://www.bsr.de/',
   'https://www.brb-baustoffe.de/brb-recyclingplaetze/berlin-koepenicker-chausse-15/',
   'https://www.wasdrack.de/',
   'https://www.bsr.de/recyclinghoefe-20503.php',
   'h

In [367]:
website_cols = [
    "website", "contact:website", "operator:website"
]

In [368]:
df_selected_clean[df_selected_clean["website"].notna()][
 ["id"] + website_cols]

,id,website,contact:website,operator:website
115,442877823,https://www.bsr.de/8856.html,None,None
521,1434318202,https://www.klebs.info/,None,None
744,2525335317,https://www.fps-entsorgung.de/,None,None
942,4431732702,http://www.drk-berlin-city.de,None,None
984,4972103512,https://www.berlin-recycling.de,None,None
1367,8722198358,http://www.berlin-textilrecycling.com/,None,None
1410,8987596142,https://www.drk.de/,None,None
1480,9218731332,https://www.drk.de/kleidersammlung,https://drk-mueggelspree.de,None
1803,10079244117,https://www.bsr.de/recyclinghoefe-20503.php?cu...,None,None
2340,11717220369,https://www.platane19.de/laeden/gebrauchtwaren...,None,None


In [369]:
df_selected_clean[df_selected_clean["contact:website"].notna()][
 ["id"] + website_cols]

,id,website,contact:website,operator:website
330,823788768,None,berlin-recycling.de,None
976,4855084318,None,http://www.texaid.de,None
1196,7951636690,None,https://www.reinhardt-rohstoffe.de/,None
1306,8544127446,None,http://www.cavtex.de,None
1375,8761752958,None,https://www.berliner-stadtmission.de/sachspenden,None
1430,9134637158,None,https://www.versero.de/,None
1480,9218731332,https://www.drk.de/kleidersammlung,https://drk-mueggelspree.de,None
1552,9549407444,None,https://karl-meyer.de/,None
1565,9566349245,None,https://www.gak-textilrecycling.de/,None
1718,9984499779,None,https://www.cavtex.de/,None


In [370]:
df_selected_clean[df_selected_clean["operator:website"].notna()][
 ["id"] + website_cols]

,id,website,contact:website,operator:website
737,2512623620,None,None,https://www.karl-meyer.de


#### Apply normalization (final pipeline)

In [371]:
def normalize_website(url):
    """
    Normalize website URLs without inventing information.

    Rules:
    - Preserve existing scheme if present
    - Assume https:// only when scheme is missing
    - Leave NA values untouched
    """
    if pd.isna(url):
        return pd.NA

    url = url.strip()

    if url.startswith(("http://", "https://")):
        return url

    # OSM commonly omits scheme; https is the modern safe default
    return f"https://{url}"



In [372]:
for col in website_cols:
    df_selected_clean[col] = (
        df_selected_clean[col]
        .astype("string")
        .apply(normalize_website)
    )
df_selected_clean[df_selected_clean["contact:website"].notna()][
 ["id"] + website_cols]

,id,website,contact:website,operator:website
330,823788768,<NA>,https://berlin-recycling.de,<NA>
976,4855084318,<NA>,http://www.texaid.de,<NA>
1196,7951636690,<NA>,https://www.reinhardt-rohstoffe.de/,<NA>
1306,8544127446,<NA>,http://www.cavtex.de,<NA>
1375,8761752958,<NA>,https://www.berliner-stadtmission.de/sachspenden,<NA>
1430,9134637158,<NA>,https://www.versero.de/,<NA>
1480,9218731332,https://www.drk.de/kleidersammlung,https://drk-mueggelspree.de,<NA>
1552,9549407444,<NA>,https://karl-meyer.de/,<NA>
1565,9566349245,<NA>,https://www.gak-textilrecycling.de/,<NA>
1718,9984499779,<NA>,https://www.cavtex.de/,<NA>


In [373]:
df_selected_clean["website_combined"] = df_selected_clean.apply(
    combine_columns,
    axis=1,
    cols=website_cols
)
df_selected_clean.head()

,element,id,recycling_type,note,source,access,wheelchair,opening_hours,website,addr:floor,...,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address,contact_combined,website_combined
0,node,26867409,container,None,None,None,None,None,<NA>,None,...,[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,<NA>,<NA>,<NA>
1,node,254985049,container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,<NA>,None,...,[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,<NA>,None,...,[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,<NA>,None,...,[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,container,None,survey,None,None,None,<NA>,None,...,[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [374]:
df_selected_clean[df_selected_clean["contact:website"].notna()][
 ["id"] + website_cols + ["website_combined"]]

,id,website,contact:website,operator:website,website_combined
330,823788768,<NA>,https://berlin-recycling.de,<NA>,contact:website: https://berlin-recycling.de
976,4855084318,<NA>,http://www.texaid.de,<NA>,contact:website: http://www.texaid.de
1196,7951636690,<NA>,https://www.reinhardt-rohstoffe.de/,<NA>,contact:website: https://www.reinhardt-rohstof...
1306,8544127446,<NA>,http://www.cavtex.de,<NA>,contact:website: http://www.cavtex.de
1375,8761752958,<NA>,https://www.berliner-stadtmission.de/sachspenden,<NA>,contact:website: https://www.berliner-stadtmis...
1430,9134637158,<NA>,https://www.versero.de/,<NA>,contact:website: https://www.versero.de/
1480,9218731332,https://www.drk.de/kleidersammlung,https://drk-mueggelspree.de,<NA>,website: https://www.drk.de/kleidersammlung; c...
1552,9549407444,<NA>,https://karl-meyer.de/,<NA>,contact:website: https://karl-meyer.de/
1565,9566349245,<NA>,https://www.gak-textilrecycling.de/,<NA>,contact:website: https://www.gak-textilrecycli...
1718,9984499779,<NA>,https://www.cavtex.de/,<NA>,contact:website: https://www.cavtex.de/


In [375]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=website_cols, inplace=True)
df_selected_clean.head()

,element,id,recycling_type,note,source,access,wheelchair,opening_hours,addr:floor,covered,...,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address,contact_combined,website_combined
0,node,26867409,container,None,None,None,None,None,None,None,...,[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,<NA>,<NA>,<NA>
1,node,254985049,container,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,None,...,[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,container,None,None,None,None,None,None,None,...,[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,container,None,None,None,None,None,None,None,...,[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,container,None,survey,None,None,None,None,None,...,[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


### Group 7: Recycling Types

#### Exploratory inspection (not part of final pipeline)

In [376]:
recycling_type_cols = groups[7]
recycling_type_cols

['not_accepted_recycling_items',
 'accepted_recycling_items',
 'recycling_type',
 'material',
 'colour',
 'green',
 'waste']

In [377]:
df_selected_clean[df_selected_clean["not_accepted_recycling_items"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
12,290659186,[glass],[glass_bottles],container,None,None,None,None
302,737216262,"[clothes, cans, paper, plastic]",[glass_bottles],container,None,None,None,None
475,1256942896,"[glass, batteries, cans, paper, scrap_metal]","[clothes, shoes]",container,None,None,None,None
487,1299614073,[plastic],"[cans, paper, plastic_packaging]",container,None,None,None,None
586,1576124803,[glass],[glass_bottles],container,None,None,None,None
684,1965564034,"[glass, batteries, cans, paper, scrap_metal]","[clothes, shoes]",container,None,None,None,None
703,2148931649,[glass],[glass_bottles],container,None,None,None,None
712,2299861699,[glass],[glass_bottles],container,None,None,None,None
718,2385420167,[glass],[glass_bottles],container,None,None,None,None
737,2512623620,"[cans, aluminium]","[glass_bottles, glass_jars, glass_bottles_unkn...",container,None,None,None,None


In [378]:
df_selected_clean[df_selected_clean["accepted_recycling_items"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
0,26867409,<NA>,[glass_bottles],container,None,None,None,None
1,254985049,<NA>,[clothes],container,None,None,None,None
2,262212828,<NA>,[glass_bottles],container,None,None,None,None
3,267093410,<NA>,[glass_bottles],container,None,None,None,None
4,272619379,<NA>,[glass_bottles],container,None,None,None,None
...,...,...,...,...,...,...,...,...
2835,1437119382,<NA>,[glass_bottles],container,None,None,None,None
2838,1448681307,[glass],"[glass_bottles, clothes]",container,None,None,None,None
2839,1451298080,<NA>,[glass_bottles],container,None,None,None,None
2840,1451319774,<NA>,[glass_bottles],container,None,None,None,None


In [379]:
df_selected_clean[df_selected_clean["recycling_type"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
0,26867409,<NA>,[glass_bottles],container,None,None,None,None
1,254985049,<NA>,[clothes],container,None,None,None,None
2,262212828,<NA>,[glass_bottles],container,None,None,None,None
3,267093410,<NA>,[glass_bottles],container,None,None,None,None
4,272619379,<NA>,[glass_bottles],container,None,None,None,None
...,...,...,...,...,...,...,...,...
2837,1448156166,<NA>,<NA>,container,None,None,None,None
2838,1448681307,[glass],"[glass_bottles, clothes]",container,None,None,None,None
2839,1451298080,<NA>,[glass_bottles],container,None,None,None,None
2840,1451319774,<NA>,[glass_bottles],container,None,None,None,None


In [380]:
df_selected_clean[df_selected_clean["material"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
169,499987314,<NA>,[glass_bottles],container,steel,None,None,None
182,531221905,<NA>,"[clothes, shoes]",container,metal,None,None,None
310,764609993,<NA>,[glass_bottles],container,steel,white;green;brown,None,None
321,793020070,<NA>,[glass_bottles],container,steel,None,None,None
330,823788768,<NA>,[glass_bottles],container,steel,silver,None,None
361,853810625,<NA>,[glass_bottles],container,steel,None,None,None
420,961626597,<NA>,[glass_bottles],container,steel,None,None,None
421,961626645,<NA>,[glass_bottles],container,steel,None,None,None
542,1554089408,<NA>,[glass_bottles],container,metal,None,None,None
854,3734445710,<NA>,"[clothes, shoes]",container,steel,None,None,None


In [381]:
df_selected_clean[df_selected_clean["colour"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
310,764609993,<NA>,[glass_bottles],container,steel,white;green;brown,None,None
330,823788768,<NA>,[glass_bottles],container,steel,silver,None,None
336,831530114,<NA>,[clothes],container,None,black,None,None
740,2517677621,<NA>,[glass_bottles],container,None,green,yes,None
1016,5584924473,<NA>,[clothes],container,metal,red,None,None
1017,5584924475,<NA>,[clothes],container,metal,black,None,None
1049,6183509465,<NA>,"[clothes, shoes]",container,None,blue,None,None
1149,7454939230,<NA>,[clothes],container,None,beige,None,None
1355,8688604786,<NA>,"[clothes, shoes]",container,None,yellow,None,None
2317,11412122516,<NA>,[clothes],container,None,white,None,None


In [382]:
df_selected_clean[df_selected_clean["id"].isin([764609993	,823788768	,2517677621, 5584924473, 5584924475, 6183509465,8688604786, 11412122516	])]

,element,id,recycling_type,note,source,access,wheelchair,opening_hours,addr:floor,covered,...,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,full_address,contact_combined,website_combined
310,node,764609993,container,None,survey,None,yes,None,None,no,...,[glass_bottles],<NA>,13.447836,52.497999,Karl Meyer Rohstoffverwertung,[operator: Karl Meyer Rohstoffverwertung],<NA>,<NA>,<NA>,<NA>
330,node,823788768,container,None,None,None,None,"Mo-Sa 07:00-13:00, 15:00-20:00",None,no,...,[glass_bottles],<NA>,13.303417,52.449555,Karl Meyer Rohstoffverwertung,[operator: Karl Meyer Rohstoffverwertung],<NA>,<NA>,<NA>,contact:website: https://berlin-recycling.de
740,node,2517677621,container,None,None,None,None,None,None,None,...,[glass_bottles],<NA>,13.29873,52.535104,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1016,node,5584924473,container,None,None,None,None,None,None,None,...,[clothes],<NA>,13.394683,52.464212,Humana Kleidersammlung GmbH,[operator: Humana Kleidersammlung GmbH],<NA>,<NA>,<NA>,<NA>
1017,node,5584924475,container,None,None,None,None,None,None,None,...,[clothes],<NA>,13.410494,52.464017,Deutsches Rotes Kreuz,[operator: Deutsches Rotes Kreuz],<NA>,<NA>,<NA>,<NA>
1049,node,6183509465,container,None,None,None,None,None,None,None,...,"[clothes, shoes]",<NA>,13.374365,52.558661,Islamic Relief Kleiderkammer,[operator: Islamic Relief Kleiderkammer],<NA>,<NA>,<NA>,<NA>
1355,node,8688604786,container,None,None,yes,None,24/7,None,None,...,"[clothes, shoes]",<NA>,13.327284,52.447173,Bera Textilrecycling,[operator: Bera Textilrecycling],<NA>,<NA>,<NA>,<NA>
2317,node,11412122516,container,None,None,None,None,24/7,None,None,...,[clothes],<NA>,13.467029,52.55571,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [383]:
df_selected_clean[df_selected_clean["green"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
740,2517677621,<NA>,[glass_bottles],container,None,green,yes,None


In [384]:
df_selected_clean[df_selected_clean["waste"].notna()][
 ["id"] + recycling_type_cols]

,id,not_accepted_recycling_items,accepted_recycling_items,recycling_type,material,colour,green,waste
1494,9281237576,<NA>,[glass_bottles],container,None,None,None,glass


#### Apply normalization (final pipeline)

In [385]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=["material", "colour", "green", "waste", "recycling_type"], inplace=True)


In [386]:
df_selected_clean.head

<bound method NDFrame.head of      element          id                                               note  \
0       node    26867409                                               None   
1       node   254985049  2017-10: nicht durch Bezirk genehmigt (https:/...   
2       node   262212828                                               None   
3       node   267093410                                               None   
4       node   272619379                                               None   
...      ...         ...                                                ...   
2837     way  1448156166                                               None   
2838     way  1448681307                                               None   
2839     way  1451298080                                               None   
2840     way  1451319774                                               None   
2841     way  1452164030                                               None   

      source   access

### Group 8: Accessability

#### Exploratory inspection (not part of final pipeline)

In [387]:
group8_report = analyze_group(df_selected_clean, groups[8])
group8_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'access': 2646,
  'wheelchair': 2706,
  'obstacle:parking': 2824,
  'lit': 2841,
  'addr:floor': 2816,
  'level': 2809,
  'indoor': 2788,
  'covered': 2796,
  'count': 2836,
  'barrier': 2769},
 'unique_values_per_column': {'access': ['residents',
   'yes',
   'private',
   'customers',
   'permit',
   'permissive',
   'unknown'],
  'wheelchair': ['yes', 'limited', 'no'],
  'obstacle:parking': ['yes'],
  'lit': ['yes'],
  'addr:floor': ['0'],
  'level': ['0', '0.5'],
  'indoor': ['no', 'yes'],
  'covered': ['no', 'yes'],
  'count': ['2'],
  'barrier': ['wall', 'fence']},
 'rows_with_multiple_filled':        access wheelchair obstacle:parking   lit addr:floor level indoor  \
 169       yes        yes             None  None          0     0     no   
 182      None       None             None  None       None  None     no   
 310      None        yes             None  None       None     0     no   
 321      None       None             None  None          0     0

In [388]:
access_cols = ["access", "wheelchair", "obstacle:parking", "lit", "addr:floor", "level",
        "indoor", "covered", "count", "barrier"]

In [389]:
df_selected_clean[df_selected_clean["access"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
10,288739735,residents,None,None,None,None,None,None,None,None,None
169,499987314,yes,yes,None,None,0,0,no,no,None,None
306,761371627,yes,None,None,None,None,None,None,None,None,None
334,830077041,private,None,None,None,None,None,None,None,None,None
381,922500807,private,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
2830,1423410374,private,None,None,None,None,None,None,None,None,None
2831,1427710061,private,None,None,None,None,None,None,None,None,fence
2832,1435699832,private,None,None,None,None,None,None,None,None,None
2834,1436755940,private,None,None,None,None,None,None,None,None,None


In [390]:
df_selected_clean[df_selected_clean["wheelchair"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
28,312385354,None,yes,None,None,None,None,None,None,None,None
43,330967542,None,yes,None,None,None,None,None,None,None,None
57,345347331,None,limited,None,None,None,None,None,None,None,None
80,388414297,None,no,None,None,None,None,None,None,None,None
100,429466365,None,yes,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
2652,258868474,yes,yes,None,None,None,None,None,None,None,None
2682,412910072,None,yes,None,None,None,None,None,None,None,None
2684,439057366,private,yes,None,None,None,None,None,None,None,None
2685,442114369,None,yes,None,None,None,None,None,None,None,None


In [391]:
df_selected_clean[df_selected_clean["obstacle:parking"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
348,835805867,None,None,yes,None,None,None,None,None,None,None
397,946947315,None,None,yes,None,None,None,None,None,None,None
508,1396365798,None,None,yes,None,None,None,None,None,None,None
556,1570571941,None,None,yes,None,None,None,None,None,None,None
607,1608593892,None,None,yes,None,None,None,None,None,None,None
884,3885179108,None,yes,yes,None,None,None,None,None,None,None
1182,7810997586,None,None,yes,None,None,None,None,None,None,None
2244,11118136994,None,None,yes,None,None,None,None,None,None,None
2245,11118136995,None,None,yes,None,None,None,None,None,None,None
2246,11118136996,None,None,yes,None,None,None,None,None,None,None


In [392]:
df_selected_clean[df_selected_clean["lit"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
1170,7694824640,None,None,None,yes,None,None,None,None,None,None


In [393]:
df_selected_clean[df_selected_clean["addr:floor"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
169,499987314,yes,yes,None,None,0,0,no,no,None,None
321,793020070,None,None,None,None,0,0,no,no,None,None
361,853810625,None,None,None,None,0,0,no,no,None,None
420,961626597,None,yes,None,None,0,0,no,no,None,None
421,961626645,None,yes,None,None,0,0,no,no,None,None
854,3734445710,None,None,None,None,0,0,no,no,None,None
895,3993976758,None,None,None,None,0,0,no,no,None,None
997,5182033085,None,None,None,None,0,0,no,no,None,None
1136,7316084881,None,None,None,None,0,None,no,no,None,None
2134,10773586304,None,None,None,None,0,0,no,no,None,None


In [394]:
df_selected_clean[df_selected_clean["level"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
169,499987314,yes,yes,None,None,0,0,no,no,None,None
310,764609993,None,yes,None,None,None,0,no,no,None,None
321,793020070,None,None,None,None,0,0,no,no,None,None
330,823788768,None,None,None,None,None,0,no,no,None,None
361,853810625,None,None,None,None,0,0,no,no,None,None
420,961626597,None,yes,None,None,0,0,no,no,None,None
421,961626645,None,yes,None,None,0,0,no,no,None,None
854,3734445710,None,None,None,None,0,0,no,no,None,None
895,3993976758,None,None,None,None,0,0,no,no,None,None
997,5182033085,None,None,None,None,0,0,no,no,None,None


In [395]:
df_selected_clean[df_selected_clean["indoor"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
169,499987314,yes,yes,None,None,0,0,no,no,None,None
182,531221905,None,None,None,None,None,None,no,no,2,None
310,764609993,None,yes,None,None,None,0,no,no,None,None
321,793020070,None,None,None,None,0,0,no,no,None,None
330,823788768,None,None,None,None,None,0,no,no,None,None
361,853810625,None,None,None,None,0,0,no,no,None,None
420,961626597,None,yes,None,None,0,0,no,no,None,None
421,961626645,None,yes,None,None,0,0,no,no,None,None
542,1554089408,None,None,None,None,None,None,no,no,None,None
854,3734445710,None,None,None,None,0,0,no,no,None,None


In [396]:
df_selected_clean[df_selected_clean["covered"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
169,499987314,yes,yes,None,None,0,0,no,no,None,None
182,531221905,None,None,None,None,None,None,no,no,2,None
310,764609993,None,yes,None,None,None,0,no,no,None,None
321,793020070,None,None,None,None,0,0,no,no,None,None
330,823788768,None,None,None,None,None,0,no,no,None,None
361,853810625,None,None,None,None,0,0,no,no,None,None
420,961626597,None,yes,None,None,0,0,no,no,None,None
421,961626645,None,yes,None,None,0,0,no,no,None,None
542,1554089408,None,None,None,None,None,None,no,no,None,None
854,3734445710,None,None,None,None,0,0,no,no,None,None


In [397]:
df_selected_clean[df_selected_clean["count"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
182,531221905,None,None,None,None,None,None,no,no,2,None
481,1289505019,None,None,None,None,None,None,None,None,2,None
482,1289968044,None,None,None,None,None,None,None,None,2,None
846,3726584213,None,None,None,None,None,None,None,None,2,None
875,3877187453,None,None,None,None,None,None,None,None,2,None
876,3877187454,None,None,None,None,None,None,None,None,2,None


In [398]:
df_selected_clean[df_selected_clean["barrier"].notna()][
 ["id"] + access_cols]

,id,access,wheelchair,obstacle:parking,lit,addr:floor,level,indoor,covered,count,barrier
2601,28198135,None,yes,None,None,0,0,None,None,None,wall
2603,29249002,private,None,None,None,None,None,None,None,None,fence
2604,39858952,private,None,None,None,None,None,None,None,None,fence
2605,43081843,private,None,None,None,None,None,None,None,None,fence
2606,45170253,private,None,None,None,None,None,None,None,None,fence
...,...,...,...,...,...,...,...,...,...,...,...
2821,1417260623,None,None,None,None,None,None,None,None,None,fence
2822,1417272053,None,None,None,None,None,None,None,None,None,fence
2823,1417272054,None,None,None,None,None,None,None,None,None,fence
2831,1427710061,private,None,None,None,None,None,None,None,None,fence


#### Apply normalization (final pipeline)

In [399]:
# access describes legal or social access restrictions
# We preserve it verbatim because values like "residents" or "permit"
# are meaningful and should not be collapsed.
df_selected_clean["access_restriction"] = (
    df_selected_clean["access"]
    .astype("string")
)


In [400]:
# Wheelchair access is a first-class accessibility feature in OSM.
# Missing values are treated as NA not "no".
df_selected_clean["wheelchair_access"] = (
    df_selected_clean["wheelchair"]
    .astype("string")
    .fillna(pd.NA)
)



In [401]:
def build_physical_obstacles(row):
    """
    Consolidate all physical obstruction signals into a single field.

    This reduces column count while preserving explicit obstacle information.
    """
    obstacles = []

    if row["obstacle:parking"] == "yes":
        obstacles.append("parking obstruction")

    if pd.notna(row["barrier"]):
        obstacles.append(row["barrier"])

    return ", ".join(obstacles) if obstacles else pd.NA


In [402]:
df_selected_clean["physical_obstacles"] = (
    df_selected_clean.apply(build_physical_obstacles, axis=1)
    .astype("string")
)

In [403]:
def build_environmental_features(row):
    """
    Consolidate environmental accessibility modifiers.

    These features affect usability but are not access restrictions.
    """
    features = []

    if row["lit"] == "yes":
        features.append("lit")

    if row["indoor"] == "yes":
        features.append("indoor")
    elif row["covered"] == "yes":
        features.append("covered")

    return ", ".join(features) if features else pd.NA


In [404]:
df_selected_clean["environmental_features"] = (
    df_selected_clean.apply(build_environmental_features, axis=1)
    .astype("string")
)


In [405]:
# addr:floor and level represent the same concept at different granularities.
# We prefer addr:floor when available.
df_selected_clean["floor_level"] = (
    df_selected_clean[["addr:floor", "level"]]
    .bfill(axis=1)
    .iloc[:, 0]
    .astype("Float64")
)


In [406]:
# Capacity is relevant for accessibility and congestion.
df_selected_clean["unit_count"] = (
    df_selected_clean["count"]
    .astype("Int64")
)


In [407]:
df_selected_clean.head()

,element,id,note,source,access,wheelchair,opening_hours,addr:floor,covered,indoor,...,entity_type,full_address,contact_combined,website_combined,access_restriction,wheelchair_access,physical_obstacles,environmental_features,floor_level,unit_count
0,node,26867409,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,node,254985049,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,None,survey,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [408]:
def build_accessibility_features(row):
    """
    Create a concise, human-readable accessibility summary.
    Derived only from explicit consolidated fields.
    """
    parts = []
    # --- Wheelchair accessibility ---

    if pd.notna(row["wheelchair_access"]):
        if row["wheelchair_access"] == "yes":
            parts.append("Wheelchair accessible")
        elif row["wheelchair_access"] == "limited":
            parts.append("Limited wheelchair access")
        elif row["wheelchair_access"] == "no":
            parts.append("Not wheelchair accessible")

    # --- Environmental features ---
    if pd.notna(row["environmental_features"]):
        parts.append(row["environmental_features"].capitalize())

    # --- Physical obstacles ---
    if pd.notna(row["physical_obstacles"]):
        parts.append(f"Obstacles: {row['physical_obstacles']}")

    # --- Access restrictions ---
    # Only mention restrictions if explicitly present and meaningful
    if pd.notna(row["access_restriction"]) and row["access_restriction"] != "yes":
        parts.append(f"Access restricted ({row['access_restriction']})")

    # Return NA if no features were derived
    return ", ".join(parts) if parts else pd.NA

In [409]:
df_selected_clean["accessibility_features"] = (
    df_selected_clean.apply(build_accessibility_features, axis=1)
    .astype("string")
)


In [410]:
df_selected_clean[df_selected_clean["accessibility_features"].notna()]

,element,id,note,source,access,wheelchair,opening_hours,addr:floor,covered,indoor,...,full_address,contact_combined,website_combined,access_restriction,wheelchair_access,physical_obstacles,environmental_features,floor_level,unit_count,accessibility_features
10,node,288739735,None,None,residents,None,None,None,None,None,...,<NA>,<NA>,<NA>,residents,<NA>,<NA>,<NA>,<NA>,<NA>,Access restricted (residents)
28,node,312385354,None,None,None,yes,None,None,None,None,...,Olivaer Platz ggü. Nr. 12,<NA>,<NA>,<NA>,yes,<NA>,<NA>,<NA>,<NA>,Wheelchair accessible
43,node,330967542,None,None,None,yes,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,yes,<NA>,<NA>,<NA>,<NA>,Wheelchair accessible
57,node,345347331,None,None,None,limited,24/7,None,None,None,...,<NA>,<NA>,<NA>,<NA>,limited,<NA>,<NA>,<NA>,<NA>,Limited wheelchair access
80,node,388414297,None,None,None,no,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,no,<NA>,<NA>,<NA>,<NA>,Not wheelchair accessible
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2832,way,1435699832,None,None,private,None,None,None,None,None,...,<NA>,<NA>,<NA>,private,<NA>,<NA>,<NA>,<NA>,<NA>,Access restricted (private)
2834,way,1436755940,None,None,private,None,None,None,None,None,...,<NA>,<NA>,<NA>,private,<NA>,<NA>,<NA>,<NA>,<NA>,Access restricted (private)
2835,way,1437119382,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,parking obstruction,<NA>,<NA>,<NA>,Obstacles: parking obstruction
2840,way,1451319774,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,parking obstruction,<NA>,<NA>,<NA>,Obstacles: parking obstruction


In [411]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=access_cols, inplace=True)
df_selected_clean.head()

,element,id,note,source,opening_hours,access:conditional,fixme,status,collection_times,landuse,...,full_address,contact_combined,website_combined,access_restriction,wheelchair_access,physical_obstacles,environmental_features,floor_level,unit_count,accessibility_features
0,node,26867409,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,node,254985049,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,None,None,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,None,survey,None,None,None,None,None,None,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


### Group 9: Opening Hours

#### Exploratory inspection (not part of final pipeline)

In [412]:
group9_report = analyze_group(df_selected_clean, groups[9])
group9_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'opening_hours': 2647,
  'collection_times': 2841,
  'access:conditional': 2840},
 'unique_values_per_column': {'opening_hours': ['24/7',
   'Mo-Sa 07:00-13:00,15:00-20:00',
   'Mo-Sa 07:00-13:00, 15:00-20:00',
   'Mo-We 07:00-17:00; Th 09:30-19:30; Fr 07:00-17:00; Sa 07:00-15:30',
   'Mo-Sa 07:00-13:00,15:00-20:00; PH closed',
   'Mo-Sa 07:00-20:00; PH closed',
   'Mo-Fr 07:00-13:00,15:00-20:00; PH closed',
   'Mo-Sa 07:00-13:00, 15:00-20:00; Su,PH off',
   'Mo-Fr 07:00-16:00',
   'Mo-Sa 07:00-13:00, 15:00-20:00; PH off',
   'Mo-Sa 07:00-13:00, 15:00-20:00; Su, PH off',
   'Mo-Sa 07:00-13:00,15:00-20:00, So,PH off',
   'Mo-Fr 10:00-16:00; Sa 10:00-13:00',
   'Mo-Fr 07:00-13:00,15:00-20:00',
   'Mo-Sa 07:00-13:00,15:00-20:00; PH off',
   'Mo-Sa 07:00-13:00, Mo-Sa 15:00-20:00; PH off',
   'Mo-We,Fr 07:00-17:00; Th 09:30-19:30; Sa 07:00-15:30',
   'Mo-Sa 07:00-13:00,15:00-20:00; Su,PH off',
   'Mo-Fr 10:00-16:00',
   'Mo-Sa 07:00-19:00',
   'Mo-Fr 09:00-19:00; Sa 

In [413]:
opening_times_cols = ["opening_hours", "collection_times", "access:conditional"]

In [414]:
df_selected_clean[df_selected_clean["opening_hours"].notna()][
 ["id"] + opening_times_cols]

,id,opening_hours,collection_times,access:conditional
57,345347331,24/7,None,None
100,429466365,"Mo-Sa 07:00-13:00,15:00-20:00",None,None
104,434345978,"Mo-Sa 07:00-13:00, 15:00-20:00",None,None
115,442877823,Mo-We 07:00-17:00; Th 09:30-19:30; Fr 07:00-17...,None,None
148,472505334,"Mo-Sa 07:00-13:00,15:00-20:00",None,None
...,...,...,...,...
2758,1311349821,24/7,None,None
2759,1311349822,24/7,None,None
2760,1311349823,24/7,None,None
2770,1317973378,24/7,None,None


In [415]:
df_selected_clean[df_selected_clean["collection_times"].notna()][
 ["id"] + opening_times_cols]

,id,opening_hours,collection_times,access:conditional
2599,10734959,Mo-Fr 09:00-19:00; Sa 07:00-14:30,Mo-Fr 09:00-19:00; Sa 07:00-14:30; Su off,None


In [416]:
df_selected_clean[df_selected_clean["access:conditional"].notna()][
 ["id"] + opening_times_cols]

,id,opening_hours,collection_times,access:conditional
1006,5311913077,24/7,None,"no @ (13:00-15:00, 20:00-07:00)"
1724,9988903473,24/7,None,"no @ (13:00-15:00, 20:00-07:00)"


#### Apply normalization (final pipeline)

In [417]:
def build_availability_info(row):
    """
    Consolidates opening hours, collection times, and conditional access
    into a single structured, human-readable string.

    Design principles:
    - No invented data
    - NA-first (returns pd.NA if nothing meaningful exists)
    - Preserves semantic distinctions via labeled segments
    """

    parts = []

    # --- Primary availability ---
    # opening_hours is the canonical OSM tag for general accessibility.
    if pd.notna(row["opening_hours"]):
        parts.append(f"hours={row['opening_hours']}")

    # --- Fallback availability ---
    # collection_times are operational (not user access),
    # but included ONLY if opening_hours is missing to avoid data loss.
    elif pd.notna(row["collection_times"]):
        parts.append(f"hours={row['collection_times']} (collection)")

    # --- Conditional restrictions ---
    # access:conditional modifies availability and must never be merged
    # into hours to avoid misinterpretation.
    if pd.notna(row["access:conditional"]):
        parts.append(f"restrictions={row['access:conditional']}")

    # Return NA if no availability-related information exists
    return " | ".join(parts) if parts else pd.NA


In [418]:
df_selected_clean["availability_info"] = (
    df_selected_clean
    .apply(build_availability_info, axis=1)
    .astype("string")
)


In [419]:
df_selected_clean[df_selected_clean["availability_info"].notna()][
 ["id"] + opening_times_cols+ ["availability_info"]]

,id,opening_hours,collection_times,access:conditional,availability_info
57,345347331,24/7,None,None,hours=24/7
100,429466365,"Mo-Sa 07:00-13:00,15:00-20:00",None,None,"hours=Mo-Sa 07:00-13:00,15:00-20:00"
104,434345978,"Mo-Sa 07:00-13:00, 15:00-20:00",None,None,"hours=Mo-Sa 07:00-13:00, 15:00-20:00"
115,442877823,Mo-We 07:00-17:00; Th 09:30-19:30; Fr 07:00-17...,None,None,hours=Mo-We 07:00-17:00; Th 09:30-19:30; Fr 07...
148,472505334,"Mo-Sa 07:00-13:00,15:00-20:00",None,None,"hours=Mo-Sa 07:00-13:00,15:00-20:00"
...,...,...,...,...,...
2758,1311349821,24/7,None,None,hours=24/7
2759,1311349822,24/7,None,None,hours=24/7
2760,1311349823,24/7,None,None,hours=24/7
2770,1317973378,24/7,None,None,hours=24/7


In [420]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=opening_times_cols, inplace=True)


In [421]:
df_selected_clean.head(10)

,element,id,note,source,fixme,status,landuse,geometry,accepted_recycling_items,not_accepted_recycling_items,...,contact_combined,website_combined,access_restriction,wheelchair_access,physical_obstacles,environmental_features,floor_level,unit_count,accessibility_features,availability_info
0,node,26867409,None,None,None,None,None,POINT (13.29683 52.50133),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,node,254985049,2017-10: nicht durch Bezirk genehmigt (https:/...,None,None,None,None,POINT (13.32054 52.484),[clothes],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,None,None,None,None,None,POINT (13.48568 52.51902),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,None,None,None,None,None,POINT (13.59289 52.50846),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,None,survey,None,None,None,POINT (13.45563 52.52183),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,node,272619880,None,None,None,None,None,POINT (13.45352 52.52585),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,node,273174898,None,None,None,None,None,POINT (13.29848 52.45765),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,node,280752631,None,None,None,None,None,POINT (13.57638 52.55336),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,node,280938557,None,None,None,None,None,POINT (13.47945 52.52635),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,node,281962480,Wegen Baustelle temporär (?) entfernt,None,None,None,None,POINT (13.45169 52.54738),[glass_bottles],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


### Group 10: Miscellaneous

#### Exploratory inspection (not part of final pipeline)

In [422]:
group10_report = analyze_group(df_selected_clean, groups[10])
group10_report

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/1273145470.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  filled_mask = subset.applymap(is_filled)


{'missing_per_column': {'source': 2712,
  'note': 2825,
  'fixme': 2839,
  'status': 2840,
  'landuse': 2827},
 'unique_values_per_column': {'source': ['survey',
   'aerowest',
   'Bezirksamt Charlottenburg-Wilmersdorf von Berlin, Umweltamt',
   'changeset 91932395',
   'chageset 91932395',
   'SenStadtU DOP10-C 2011 Erlaubnis 1002-12',
   'gps',
   'bing',
   'local knowledge',
   'Garmin Oregon 550',
   'Geoportal Berlin / Hauskoordinaten',
   'changeset 82438627',
   'changeset 84916889',
   'local_knowledge',
   'changeset 100950997',
   'changeset 101406521',
   'changeset 106781524',
   'changeset 120272901',
   'changeset 138479910',
   'changeset 154662730',
   'survey;local knowledge'],
  'note': ['2017-10: nicht durch Bezirk genehmigt (https://www.berlin.de/ba-charlottenburg-wilmersdorf/verwaltung/aemter/ordnungsamt/altkleidercontainer-in-charlottenburg-wilmersdorf-520269.php). Da ggf. auf Privatgrund aber nicht entfernt?',
   'Wegen Baustelle temporär (?) entfernt',
   'Stan

#### Apply normalization (final pipeline)

In [423]:
# Normalize common source spelling inconsistencies
df_selected_clean["source"] = (
    df_selected_clean["source"]
    .str.lower()
    .str.replace("chageset", "changeset", regex=False)
    .str.strip()
    .astype("string")
)


In [424]:
# Drop free-text editorial metadata that does not describe the POI itself
df_selected_clean.drop(columns=["note", "fixme"], inplace=True)


In [425]:
# Map operational status to a clear, NA-safe flag
df_selected_clean["is_operational"] = (
    df_selected_clean["status"]
    .map({
        "functional": True,
        "closed": False
    })
    .astype("boolean")
)

df_selected_clean.drop(columns=["status"], inplace=True)


In [426]:
df_selected_clean["landuse"] = (
    df_selected_clean["landuse"]
    .astype("string")
)


In [427]:
df_selected_clean.head()

,element,id,source,landuse,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,...,website_combined,access_restriction,wheelchair_access,physical_obstacles,environmental_features,floor_level,unit_count,accessibility_features,availability_info,is_operational
0,node,26867409,<NA>,<NA>,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,node,254985049,<NA>,<NA>,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,node,262212828,<NA>,<NA>,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,node,267093410,<NA>,<NA>,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,node,272619379,survey,<NA>,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Adding district Information

In [428]:
import os
print(os.getcwd())


/Users/sugar/Documents/GitHub/Webeet/layered-populate-data-pool-da/recycling_points/scripts


In [429]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("../../mapping/lor_ortsteile.geojson")

In [430]:
berlin_districts_gdf.head()

,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA,geometry
0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,0102,Moabit,Polygon,Moabit,Mitte,768.7909,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,0103,Hansaviertel,Polygon,Hansaviertel,Mitte,52.5337,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,0104,Tiergarten,Polygon,Tiergarten,Mitte,516.0672,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,0105,Wedding,Polygon,Wedding,Mitte,919.9112,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


In [431]:

# Create a dedicated geometry for spatial join
df_selected_clean["join_geometry"] = df_selected_clean.geometry

df_selected_clean.loc[polygon_mask, "join_geometry"] = (
    df_selected_clean.loc[polygon_mask, "geometry"].centroid
)

df_selected_clean = df_selected_clean.set_geometry("join_geometry")

/var/folders/_f/4wp7s_s1233ggbnpxfl1d12r0000gn/T/ipykernel_31142/3112141280.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_selected_clean.loc[polygon_mask, "geometry"].centroid


In [432]:
# Spatial join
recycling_point_df_district = gpd.sjoin(
    df_selected_clean,
    berlin_districts_gdf[["BEZIRK", "OTEIL", "spatial_name", "geometry"]],
    how="left",
    predicate="within"
)

In [433]:
recycling_point_df_district.head()

,element,id,source,landuse,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,...,floor_level,unit_count,accessibility_features,availability_info,is_operational,join_geometry,index_right,BEZIRK,OTEIL,spatial_name
0,node,26867409,<NA>,<NA>,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.29683 52.50133),21,Charlottenburg-Wilmersdorf,Charlottenburg,0401
1,node,254985049,<NA>,<NA>,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.32054 52.484),22,Charlottenburg-Wilmersdorf,Wilmersdorf,0402
2,node,262212828,<NA>,<NA>,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.48568 52.51902),77,Lichtenberg,Lichtenberg,1103
3,node,267093410,<NA>,<NA>,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.59289 52.50846),72,Marzahn-Hellersdorf,Kaulsdorf,1003
4,node,272619379,survey,<NA>,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.45563 52.52183),6,Friedrichshain-Kreuzberg,Friedrichshain,0201


In [434]:
##just renaming columns for proper schema
recycling_point_df_district = recycling_point_df_district.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
}).drop(columns=["index_right"])  # drop district_number if not needed
recycling_point_df_district.head()

,element,id,source,landuse,geometry,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,...,environmental_features,floor_level,unit_count,accessibility_features,availability_info,is_operational,join_geometry,district,neighborhood,neighborhood_id
0,node,26867409,<NA>,<NA>,POINT (13.29683 52.50133),[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.29683 52.50133),Charlottenburg-Wilmersdorf,Charlottenburg,0401
1,node,254985049,<NA>,<NA>,POINT (13.32054 52.484),[clothes],<NA>,13.320537,52.483997,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.32054 52.484),Charlottenburg-Wilmersdorf,Wilmersdorf,0402
2,node,262212828,<NA>,<NA>,POINT (13.48568 52.51902),[glass_bottles],<NA>,13.485681,52.519021,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.48568 52.51902),Lichtenberg,Lichtenberg,1103
3,node,267093410,<NA>,<NA>,POINT (13.59289 52.50846),[glass_bottles],<NA>,13.592894,52.508457,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.59289 52.50846),Marzahn-Hellersdorf,Kaulsdorf,1003
4,node,272619379,survey,<NA>,POINT (13.45563 52.52183),[glass_bottles],<NA>,13.455634,52.521829,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.45563 52.52183),Friedrichshain-Kreuzberg,Friedrichshain,0201


In [435]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
recycling_point_df_district['district_id'] = recycling_point_df_district['district'].map(district_mapping).astype(str)

# (Optional) Check if some districts were not mapped
unmapped = recycling_point_df_district[~recycling_point_df_district['district'].isin(district_mapping.keys())]['district'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts found:", unmapped)


## Final Dataset Characteristics

The resulting GeoDataFrame contains:
- One row per recycling location
- Human-readable name and address
- Explicit accepted / rejected recycling materials
- Normalized contact and website information
- Geometry preserved for spatial analysis
- District mapping


In [ ]:
recycling_point_df_district.drop(columns=["element", "geometry"], inplace=True) #not needed as part of index in final output and the join_geometry is used asthe geometry

In [447]:
recycling_point_df_district.head()

,id,source,landuse,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,display_name,name_metadata,entity_type,...,floor_level,unit_count,accessibility_features,availability_info,is_operational,join_geometry,district,neighborhood,neighborhood_id,district_id
0,26867409,<NA>,<NA>,[glass_bottles],<NA>,13.296828,52.501329,Berlin Recycling,[operator: Berlin Recycling],<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.29683 52.50133),Charlottenburg-Wilmersdorf,Charlottenburg,0401,11004004
1,254985049,<NA>,<NA>,[clothes],<NA>,13.320537,52.483997,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.32054 52.484),Charlottenburg-Wilmersdorf,Wilmersdorf,0402,11004004
2,262212828,<NA>,<NA>,[glass_bottles],<NA>,13.485681,52.519021,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.48568 52.51902),Lichtenberg,Lichtenberg,1103,11011011
3,267093410,<NA>,<NA>,[glass_bottles],<NA>,13.592894,52.508457,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.59289 52.50846),Marzahn-Hellersdorf,Kaulsdorf,1003,11010010
4,272619379,survey,<NA>,[glass_bottles],<NA>,13.455634,52.521829,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,POINT (13.45563 52.52183),Friedrichshain-Kreuzberg,Friedrichshain,0201,11002002


## Exporting processed data to geojson and csv

In [449]:
recycling_point_df_district.to_file("../sources/final_recycling_points_with_district.geojson", driver="GeoJSON")

In [450]:
recycling_point_df_district.to_csv("../sources/final_recycling_points_with_district.csv", index=False)

In [451]:
recycling_point_df_district["district_id"].isna().sum()


np.int64(0)

In [452]:
recycling_point_df_district.geometry.geom_type.value_counts()


Point    2842
Name: count, dtype: int64

In [454]:
recycling_point_df_district["full_address"].isna().sum()


np.int64(2753)